# Document OCR Pipeline — Quantization

**GPTQ · AWQ · SmoothQuant · SpinQuant · ConvRot**

*Part 2 of 5 · Shrinking $f_\theta$ without breaking OCR*

📎 **[Phases A→D — complete function journey](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phases_abcd_journey.png)**

Shrink **Florence-2** for document OCR using five post-training quantization methods — all implemented **by hand in PyTorch**. No `auto-gptq`, `awq`, `bitsandbytes`, or `optimum-quanto`.

**Model:** `microsoft/Florence-2-base-ft` · **Task:** `<OCR_WITH_REGION>` detect

---

## Table of contents

| Section | Stages | Run order |
|---------|--------|-----------|
| **0 — Install & config** | config cell | 1st |
| **1–6 — Theory** | building blocks + one stage per method | 2nd |
| **7 — Load model** | Florence-2 + calibration image | 3rd |
| **A — Baseline** | 8a–8c | 4th |
| **B — Survey** | 9a–9b | 5th |
| **C — Sensitivity** | 10a–10b | 6th |
| **D — Smart quant** | 11–14 | 7th |
| **E — Deep dives** | 15–16 (optional) | last |

**Run Phases A → D in order.** Each phase has markdown (what/why) then code (do it).

---

## Why quantize?

A linear layer stores weight matrix $\mathbf{W} \in \mathbb{R}^{O \times I}$. In fp16 that is $2OI$ bytes. With $b$-bit symmetric quantization:

$$
Q(\mathbf{W}) = \text{round}\!\left(\frac{\mathbf{W}}{s}\right), \quad s = \frac{\max |\mathbf{W}|}{2^{b-1}-1}, \quad
\hat{\mathbf{W}} = Q(\mathbf{W}) \cdot s
$$

For $b=4$, storage drops $\approx 4\times$ vs fp16. The hard part is choosing $Q$ so $\|\mathbf{W} - \hat{\mathbf{W}}\|$ stays small **in the directions calibration data actually uses**.

| Phase | Stages | What you learn |
|-------|--------|----------------|
| **A — Baseline** | 8a–8c | Naive int4 everywhere → `baseline_ocr` |
| **B — Survey** | 9a–9b | Layer inventory → `profiles_b` |
| **C — Sensitivity** | 10a–10b | Rank layers → `profiles_c` |
| **D — Smart quant** | 11–14 | Mixed-precision plan → beat Phase A |
| **E — Deep dives** | 15–16 | Compare all five methods on OCR |

![Why each phase has its own LayerProfile class](attachment:nb02_classes_layerprofile_flow.png)

📎 **[All 23 classes — map (what + why)](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_all_classes_map.png)**

📎 **[Why 5 LayerProfile classes — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_why_five_layerprofiles.png)**

Each phase redefines `LayerProfile` with different fields. Pick **`QUANT_METHOD`** in the config cell and turn on **`MIXED_PRECISION`** for Phase D. **GPU recommended.**

> **Diagrams:** Stage overview images are **embedded in this notebook** so they render on GitHub (including private repos). Line-by-line walkthroughs are **clickable links** (📎) — click to open the full diagram.

---

## Series

[01 OCR](https://github.com/Gaurav14cs17/Document-OCR-Pipeline/blob/main/01_document_ocr_pipeline.ipynb) → **02 Quant** (`ocr_pipeline_quant.ipynb`) → [03 Mobile export](https://github.com/Gaurav14cs17/Document-OCR-Pipeline/blob/main/03_ocr_pipeline_mobile.ipynb) → [04 Mobile complete](https://github.com/Gaurav14cs17/Document-OCR-Pipeline/blob/main/04_ocr_pipeline_mobile_complete.ipynb) → [05 Production issues](https://github.com/Gaurav14cs17/Document-OCR-Pipeline/blob/main/05_mobile_production_issues.ipynb)


## 0 — Install & config

Quantization is **post-training**: we freeze $\theta$ and replace selected $\mathbf{W}$ with $\hat{\mathbf{W}}$. Calibration data $\{\mathbf{X}^{(m)}\}_{m=1}^M$ from real OCR forwards supplies the statistics each method needs.

**Steps:**

1. Set **`QUANT_METHOD`** (`gptq` | `awq` | `smoothquant` | `spinquant` | `convrot`) — or use **Stage 16** to compare all five.
2. Set **`MIXED_PRECISION = True`** to auto-pick int4 / int8 / fp16 per layer (recommended).
3. Run the install cell. If Colab restarts, click **Run all**.

**Plain deps only** — no quant libraries. That forces us to implement the math, not hide it behind APIs.


In [ ]:
import os
import re
import subprocess
import sys

# ── CONFIG — change these before you run ─────────────────

# ── General ──
QUANT_METHOD = "gptq"          # gptq | awq | smoothquant | spinquant | convrot
MIXED_PRECISION = True         # True = int4/int8/fp16 per layer; False = one BITS everywhere
BITS = 4                       # used only when MIXED_PRECISION = False
TASK = "detect"                # detect | ocr — sanity check after quant
MODEL_ID = "microsoft/Florence-2-base-ft"

# ── Calibration ──
MAX_CALIB_BATCHES = 8          # OCR passes for calibration + sensitivity
MAX_QUANT_LAYERS = None        # None = all quantizable layers (Linear/Conv/Embedding); cap for quick tests
MAX_ANALYZE_LAYERS = None      # None = analyze all; lower number for faster Colab runs

# ── Mixed-precision policy ──
FP16_SENSITIVE_PCT = 15        # top 15% most sensitive → keep fp16
INT8_MID_PCT = 35              # next 35% → int8; rest → int4
MIN_PARAMS_TO_QUANT = 4096     # skip tiny layers (not worth quantizing)
ALWAYS_FP16_PATTERNS = (       # never quantize layers whose name contains:
    "lm_head", "embed", "pooler", "classifier",
    "visual_projection", "image_projection", "vision_tower",
)

# ── GPTQ ──
GPTQ_BLOCK_SIZE = 128          # column block size for Hessian update
GPTQ_DAMPING = 0.01            # diagonal damping for numerical stability

# ── AWQ ──
AWQ_GRID_STEPS = 20            # scale search grid resolution

# ── SmoothQuant ──
SMOOTHQUANT_ALPHA = 0.5        # migration strength: 0=all on weights, 1=all on activations

# ── SpinQuant ──
SPINQUANT_REFINE_STEPS = 20    # Cayley/Givens refinement steps on calibration data
SPINQUANT_LR = 0.05            # learning rate for rotation optimization
SPINQUANT_N_GIVENS = 48        # number of learned Givens rotations
SPINQUANT_MAX_DIM = 1024       # skip full learn if in_features larger (use fewer Givens)

# ── ConvRot ──
CONVROT_GROUP_SIZE = 256       # RHT block N_0 — power of 4: 16, 64, 256, 1024

# ── Stage 16 comparison ──
RUN_ALL_METHODS_OCR = True     # compare all five quant methods on full OCR
COMPARE_LAYER_MSE = True       # weight+output MSE on one layer for all 5 methods

def _pip_version(package):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "show", package],
        capture_output=True, text=True, check=False,
    )
    m = re.search(r"^Version: (.+)$", r.stdout, re.M)
    return m.group(1) if m else ""

def _restart_runtime():
    print("Restarting runtime so the correct transformers version loads...")
    os.kill(os.getpid(), 9)

def ensure_transformers():
    target = "4.49.0"
    ok = lambda v: v.startswith("4.49")
    pip_ver = _pip_version("transformers")
    if not ok(pip_ver):
        print(f"Installing transformers {target} (pip had {pip_ver or 'none'})...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--force-reinstall", "transformers==4.49.0",
        ])
        pip_ver = _pip_version("transformers")
        if not ok(pip_ver):
            raise RuntimeError(f"Could not install transformers {target}")
    try:
        import transformers
        loaded = transformers.__version__
    except ImportError:
        loaded = None
    if loaded and not ok(loaded):
        print(f"pip has {pip_ver} but Python still has {loaded}. Restarting...")
        _restart_runtime()
    return pip_ver

# Plain deps only — no quant libraries
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26", "scipy>=1.12", "scikit-learn",
    "torch", "accelerate", "pillow", "matplotlib", "requests", "huggingface_hub",
])

_tf_ver = ensure_transformers()
print(f"Ready — {QUANT_METHOD.upper()}  |  mixed_precision={MIXED_PRECISION}  |  transformers {_tf_ver}")
print("All quant code is plain PyTorch — no auto-gptq, awq, bitsandbytes, or quanto.")


---
## Stage 1 — Building blocks

![Stage 1 — symmetric int4 building blocks](attachment:nb02_stage01_building_blocks.png)

Shared primitives for all five quantization methods.

### Symmetric quantization

📎 **[symmetric_quantize_per_channel — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_symmetric_quantize_lines.png)**



For bit width $b$, quantile grid $\mathcal{Q}_b = \{-2^{b-1}, \ldots, 2^{b-1}-1\}$:

$$
s = \frac{\max |w|}{2^{b-1}-1}, \quad q = \Pi_{\mathcal{Q}_b}\!\left(\text{round}\!\left(\frac{w}{s}\right)\right), \quad \hat{w} = q \cdot s
$$

where $\Pi_{\mathcal{Q}_b}$ clips to the nearest representable integer.

**Theorem (uniform $L_\infty$ bound):** for non-clipped $w$ (i.e. $|w/s| \le 2^{b-1}-1$):

$$
|w - \hat{w}| = \left|w - s \cdot \text{round}\!\left(\frac{w}{s}\right)\right| \le \frac{s}{2} = \frac{\max|w|}{2(2^{b-1}-1)}
$$

**Proof:** write $w/s = n + \epsilon$ with $n \in \mathbb{Z}$, $|\epsilon| \le 1/2$. Then $|w - s\cdot\text{round}(w/s)| = s|\epsilon| \le s/2$.

**Quantization noise variance** (rough): $\mathbb{E}[(w-\hat{w})^2] \approx s^2/12$ under uniform $\epsilon$ (same as uniform mid-tread quantizer).

### QuantState — packed int4 storage

![QuantState — how weights are packed after quantize](attachment:nb02_class_quantstate_flow.png)

### Quantized module classes

📎 **[QuantState classes — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_quantstate_classes_lines.png)**



![Quant module classes — drop-in replacements for nn.Linear](attachment:nb02_classes_quant_modules.png)

**`nn.Linear`** — five separate classes

📎 **[WeightQuantState + WeightQuantLinear — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_weightquant_state_linear_lines.png)**

📎 **[GenericRTNQuantizer + GenericAWQQuantizer — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_generic_quantizer_lines.png)**

📎 **[QuantizedConv2d + QuantizedEmbedding — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_quantized_conv_embedding_lines.png)**


 (`GPTQLinear`, `AWQLinear`, `SmoothQuantLinear`, `SpinQuantLinear`, `ConvRotLinear`):

- **GPTQ / AWQ:** $\mathbf{y} = \mathbf{x} \cdot \text{dequant}(Q(\mathbf{W}))^\top$
- **SmoothQuant:** $\mathbf{y} = (\mathbf{x} \odot s) \cdot \text{dequant}(Q(\mathbf{W}/s))^\top$
- **SpinQuant:** $\mathbf{y} = (\mathbf{x}\mathbf{R}) \cdot \text{dequant}(Q(\mathbf{W}\mathbf{R}))^\top$
- **ConvRot:** $\mathbf{y} = \sum_i \text{RHT}(\mathbf{x}_i)\, \text{RHT}(\mathbf{W}_i)^\top$ (group size $N_0$)

**`nn.Conv1d/2d/3d` + `nn.Embedding`** — `QuantizedConv*` / `QuantizedEmbedding` with per-output-channel (or per-row) RTN/AWQ. Works on ViT/CNN vision encoders and token embeddings in any architecture.

`build_quantized_module()` picks the right drop-in class for LLM, VLM, ViT, or CNN models.

Notebook 03 replaces fake quant with integer GEMM.


In [ ]:
from __future__ import annotations

import math
import re
import time
from dataclasses import dataclass
from io import BytesIO
from typing import Callable

import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # use GPU if available for speed
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")


# ═══════════════════════════════════════════════════════════════
# Symmetric per-channel quantize / dequantize
# ═══════════════════════════════════════════════════════════════
def qmax_for_bits(n_bits: int) -> int:
    """Largest representable positive integer in signed n-bit format."""
    return 2 ** (n_bits - 1) - 1  # e.g. int4 → 7, int8 → 127


def symmetric_quantize_per_channel(W: torch.Tensor, n_bits: int = 4):
    """Quantize weight matrix W[out, in] to int with one scale per output row."""
    qmax = qmax_for_bits(n_bits)                     # max int value for this bit-width
    W = W.float()                                    # upcast to float32 for precision
    max_abs = W.abs().amax(dim=1).clamp(min=1e-8)   # largest magnitude per row (avoid /0)
    scales = max_abs / qmax                          # scale = max_abs / qmax → maps range to [-qmax, qmax]
    q = torch.round(W / scales.unsqueeze(1)).clamp(-qmax - 1, qmax).to(torch.int8)  # quantize + clip
    return q, scales


def symmetric_dequant_per_channel(q: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
    """Reverse quantization: int8 → float32 via scale multiplication."""
    return q.float() * scales.unsqueeze(1)  # broadcast scale per row


# ═══════════════════════════════════════════════════════════════
# QuantState dataclasses — one per method
# ═══════════════════════════════════════════════════════════════
@dataclass
class GPTQState:
    """GPTQ quantized state — Hessian-aware column quantization."""
    weight_q: torch.Tensor            # quantized int8 weight matrix
    weight_scales: torch.Tensor       # per-channel dequant scales
    bias: torch.Tensor | None         # original bias (unchanged)
    n_bits: int = 4                   # bit-width used
    method: str = "gptq"


@dataclass
class AWQState:
    """AWQ quantized state — activation-aware scale search."""
    weight_q: torch.Tensor            # quantized int8 weight matrix
    weight_scales: torch.Tensor       # per-channel dequant scales
    bias: torch.Tensor | None         # original bias (unchanged)
    n_bits: int = 4
    method: str = "awq"


@dataclass
class SmoothQuantState:
    """SmoothQuant state — outlier migration from activations to weights."""
    weight_q: torch.Tensor            # quantized int8 weight matrix (after smoothing)
    weight_scales: torch.Tensor       # per-channel dequant scales
    bias: torch.Tensor | None         # original bias (unchanged)
    act_scales: torch.Tensor          # per-channel scales applied to activations at inference
    n_bits: int = 4
    method: str = "smoothquant"


@dataclass
class SpinQuantState:
    """SpinQuant state — learned Givens rotation before quantization."""
    weight_q: torch.Tensor            # quantized int8 weight matrix (rotated)
    weight_scales: torch.Tensor       # per-channel dequant scales
    bias: torch.Tensor | None         # original bias (unchanged)
    givens_angles: torch.Tensor       # [K] learned rotation angles
    givens_pairs: torch.Tensor        # [K, 2] index pairs for each Givens rotation
    n_bits: int = 4
    method: str = "spinquant"


@dataclass
class ConvRotState:
    """ConvRot state — group-wise Regular Hadamard Transform."""
    weight_q: torch.Tensor            # quantized int8 weight matrix (rotated)
    weight_scales: torch.Tensor       # per-channel dequant scales
    bias: torch.Tensor | None         # original bias (unchanged)
    group_size: int                   # RHT block size N_0 (power of 4: 16/64/256/1024)
    pad: int                          # padding columns added to input dim
    n_bits: int = 4
    method: str = "convrot"


# Union type for all quantized states
@dataclass
class WeightQuantState:
    """Generic per-output-channel symmetric quant (Conv*, Embedding, RTN fallback)."""
    weight_q: torch.Tensor            # quantized int8 weights (original shape)
    weight_scales: torch.Tensor       # per output-channel scales
    weight_shape: tuple               # original weight tensor shape
    module_type: str                  # Linear | Conv2d | Embedding | ...
    bias: torch.Tensor | None = None  # optional bias
    n_bits: int = 4
    method: str = "rtn"               # rtn | awq (for non-Linear layers)


# Union type for all quantized states (Linear methods + universal fallback)
QuantState = GPTQState | AWQState | SmoothQuantState | SpinQuantState | ConvRotState | WeightQuantState

# Module types we can swap in Stages 8-12 (LLM / VLM / ViT / CNN)
QUANT_MODULE_TYPES = (nn.Linear, nn.Conv1d, nn.Conv2d, nn.Conv3d, nn.Embedding)
LINEAR_ONLY_METHODS = frozenset({"gptq", "smoothquant", "spinquant", "convrot"})


# ═══════════════════════════════════════════════════════════════
# SpinQuant: Givens rotation helper
# ═══════════════════════════════════════════════════════════════
def givens_rotation_matrix(angles: torch.Tensor, pairs: torch.Tensor, n: int, device, dtype=torch.float32):
    """Build rotation R = G_K ... G_1 from K Givens rotations (SpinQuant)."""
    R = torch.eye(n, device=device, dtype=dtype)  # start with identity
    for k in range(angles.shape[0]):
        i, j = int(pairs[k, 0]), int(pairs[k, 1])  # plane indices for this rotation
        if i == j:
            continue  # degenerate — skip
        c = torch.cos(angles[k])  # cos(theta_k)
        s = torch.sin(angles[k])  # sin(theta_k)
        Gi = torch.eye(n, device=device, dtype=dtype)  # identity modified at (i,j) entries
        Gi[i, i] = c   # rotation in the (i,j) plane
        Gi[j, j] = c
        Gi[i, j] = -s
        Gi[j, i] = s
        R = Gi @ R  # compose all rotations into single matrix
    return R


# ═══════════════════════════════════════════════════════════════
# ConvRot: group-wise Regular Hadamard Transform (RHT)
# ═══════════════════════════════════════════════════════════════
_H4_BASE = torch.tensor(
    [[1, 1, 1, -1], [1, 1, -1, 1], [1, -1, 1, 1], [-1, 1, 1, 1]], dtype=torch.float32
)  # base regular Hadamard matrix of order 4 (minimal column discrepancy)


def regular_hadamard_matrix(n: int, device, dtype=torch.float32) -> torch.Tensor:
    """Build regular H-matrix of order n=4^k via Kronecker product."""
    if n < 4:
        raise ValueError(f"ConvRot group size must be >= 4, got {n}")
    t = n
    while t % 4 == 0:  # verify n is a power of 4
        t //= 4
    if t != 1:
        raise ValueError(f"ConvRot group size must be power of 4, got {n}")
    H = _H4_BASE.to(device=device, dtype=dtype)
    while H.shape[0] < n:
        H = torch.kron(H, _H4_BASE.to(device=device, dtype=dtype))  # Kronecker product: H_{4^{k+1}} = H_{4^k} ⊗ H_4
    H = H[:n, :n] / math.sqrt(n)  # normalize so H @ H^T = I (orthogonal)
    return H


def convrot_rotation_matrix(in_features: int, group_size: int, device, dtype=torch.float32):
    """Build block-diagonal RHT matrix — O(K) complexity vs global O(K^2)."""
    pad = (group_size - in_features % group_size) % group_size  # pad to make divisible by group_size
    K = in_features + pad                                        # total dimension after padding
    H = regular_hadamard_matrix(group_size, device, dtype)       # one block of size N_0
    R = torch.block_diag(*([H] * (K // group_size)))             # repeat along diagonal
    return R, pad


def apply_convrot_rht(x: torch.Tensor, group_size: int, pad: int) -> torch.Tensor:
    """Apply group-wise RHT to activations [..., in_features]."""
    if pad:
        x = F.pad(x, (0, pad))  # zero-pad last dim to align with group boundaries
    R, _ = convrot_rotation_matrix(x.shape[-1], group_size, x.device, x.dtype)
    return x @ R  # rotate activations (smooths outliers within each group)


# ═══════════════════════════════════════════════════════════════
# GPTQ Linear — simple dequant + matmul (no activation transform)
# ═══════════════════════════════════════════════════════════════
class GPTQLinear(nn.Module):
    """Drop-in Linear for GPTQ: stores int weights, dequantizes in forward."""

    def __init__(self, in_features: int, out_features: int, state: GPTQState):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.method = "gptq"
        self.n_bits = state.n_bits
        self.register_buffer("weight_q", state.weight_q)           # int8 weights
        self.register_buffer("weight_scales", state.weight_scales) # per-row scales
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None

    @property
    def weight_fp(self) -> torch.Tensor:
        return symmetric_dequant_per_channel(self.weight_q, self.weight_scales)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = symmetric_dequant_per_channel(self.weight_q, self.weight_scales).to(x.dtype)
        return F.linear(x, w, self.bias)

    def storage_bytes(self) -> int:
        """Estimate memory: weights (int8) + scales (fp32) + bias (fp32)."""
        n = self.weight_q.numel() + self.weight_scales.numel()
        if self.bias is not None:
            n += self.bias.numel()
        return n * 4


# ═══════════════════════════════════════════════════════════════
# AWQ Linear — same as GPTQ (scales baked into weights at quant time)
# ═══════════════════════════════════════════════════════════════
class AWQLinear(nn.Module):
    """Drop-in Linear for AWQ: stores int weights, dequantizes in forward."""

    def __init__(self, in_features: int, out_features: int, state: AWQState):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.method = "awq"
        self.n_bits = state.n_bits
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None

    @property
    def weight_fp(self) -> torch.Tensor:
        return symmetric_dequant_per_channel(self.weight_q, self.weight_scales)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = symmetric_dequant_per_channel(self.weight_q, self.weight_scales).to(x.dtype)
        return F.linear(x, w, self.bias)

    def storage_bytes(self) -> int:
        """Estimate memory: weights (int8) + scales (fp32) + bias (fp32)."""
        n = self.weight_q.numel() + self.weight_scales.numel()
        if self.bias is not None:
            n += self.bias.numel()
        return n * 4


# ═══════════════════════════════════════════════════════════════
# SmoothQuant Linear — scale activations, then dequant + matmul
# ═══════════════════════════════════════════════════════════════
class SmoothQuantLinear(nn.Module):
    """Drop-in Linear for SmoothQuant: scales activations by s_j before matmul."""

    def __init__(self, in_features: int, out_features: int, state: SmoothQuantState):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.method = "smoothquant"
        self.n_bits = state.n_bits
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.register_buffer("act_scales", state.act_scales)  # per-channel activation scales
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None

    @property
    def weight_fp(self) -> torch.Tensor:
        w = symmetric_dequant_per_channel(self.weight_q, self.weight_scales)
        return w / self.act_scales.unsqueeze(0)  # undo migration to recover original W

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x * self.act_scales.to(x.dtype)  # apply channel scales to activations
        w = symmetric_dequant_per_channel(self.weight_q, self.weight_scales).to(x.dtype)
        return F.linear(x, w, self.bias)

    def storage_bytes(self) -> int:
        """Estimate memory: weights + scales + act_scales + bias."""
        n = self.weight_q.numel() + self.weight_scales.numel() + self.act_scales.numel()
        if self.bias is not None:
            n += self.bias.numel()
        return n * 4


# ═══════════════════════════════════════════════════════════════
# SpinQuant Linear — rotate activations by learned R, then dequant + matmul
# ═══════════════════════════════════════════════════════════════
class SpinQuantLinear(nn.Module):
    """Drop-in Linear for SpinQuant: rotates activations by Givens R before matmul."""

    def __init__(self, in_features: int, out_features: int, state: SpinQuantState):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.method = "spinquant"
        self.n_bits = state.n_bits
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.register_buffer("givens_angles", state.givens_angles)  # learned rotation angles
        self.register_buffer("givens_pairs", state.givens_pairs)    # rotation plane indices
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None

    def _build_rotation(self, device, dtype):
        return givens_rotation_matrix(
            self.givens_angles.to(device), self.givens_pairs.to(device),
            self.in_features, device, dtype,
        )

    @property
    def weight_fp(self) -> torch.Tensor:
        w = symmetric_dequant_per_channel(self.weight_q, self.weight_scales)
        R = self._build_rotation(w.device, w.dtype)
        return w @ R.T  # undo rotation: W_orig ≈ W_rot @ R^T

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        R = self._build_rotation(x.device, x.dtype)
        x = x @ R  # rotate activations to match rotated weights
        w = symmetric_dequant_per_channel(self.weight_q, self.weight_scales).to(x.dtype)
        return F.linear(x, w, self.bias)

    def storage_bytes(self) -> int:
        """Estimate memory: weights + scales + angles + pairs + bias."""
        n = self.weight_q.numel() + self.weight_scales.numel()
        n += self.givens_angles.numel() + self.givens_pairs.numel()
        if self.bias is not None:
            n += self.bias.numel()
        return n * 4


# ═══════════════════════════════════════════════════════════════
# ConvRot Linear — group-wise RHT on activations, then dequant + matmul
# ═══════════════════════════════════════════════════════════════
class ConvRotLinear(nn.Module):
    """Drop-in Linear for ConvRot: applies group-wise Regular Hadamard before matmul."""

    def __init__(self, in_features: int, out_features: int, state: ConvRotState):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.method = "convrot"
        self.n_bits = state.n_bits
        self.group_size = state.group_size  # RHT block size N_0
        self.pad = state.pad                # input-dim padding
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None

    @property
    def weight_fp(self) -> torch.Tensor:
        w = symmetric_dequant_per_channel(self.weight_q, self.weight_scales)
        if self.pad:
            w = F.pad(w, (0, self.pad))
        R, _ = convrot_rotation_matrix(w.shape[1], self.group_size, w.device, w.dtype)
        w = w @ R.T  # undo rotation
        if self.pad:
            w = w[:, :self.in_features]  # remove padding
        return w

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = apply_convrot_rht(x, self.group_size, self.pad)  # group-wise RHT
        w = symmetric_dequant_per_channel(self.weight_q, self.weight_scales).to(x.dtype)
        return F.linear(x, w, self.bias)

    def storage_bytes(self) -> int:
        """Estimate memory: weights + scales + bias (group_size/pad are ints, negligible)."""
        n = self.weight_q.numel() + self.weight_scales.numel()
        if self.bias is not None:
            n += self.bias.numel()
        return n * 4


# ═══════════════════════════════════════════════════════════════
# Factory: build the right QuantizedLinear from any QuantState
# ═══════════════════════════════════════════════════════════════
def QuantizedLinear(in_features: int, out_features: int, state: QuantState) -> nn.Module:
    """Factory that returns the correct quantized Linear class for the given state."""
    if isinstance(state, GPTQState):
        return GPTQLinear(in_features, out_features, state)
    if isinstance(state, AWQState):
        return AWQLinear(in_features, out_features, state)
    if isinstance(state, SmoothQuantState):
        return SmoothQuantLinear(in_features, out_features, state)
    if isinstance(state, SpinQuantState):
        return SpinQuantLinear(in_features, out_features, state)
    if isinstance(state, ConvRotState):
        return ConvRotLinear(in_features, out_features, state)
    raise ValueError(f"Unknown state type: {type(state)}")


# ═══════════════════════════════════════════════════════════════
# Helpers: reshape weights for Conv / Embedding
# ═══════════════════════════════════════════════════════════════
def module_type_name(module: nn.Module) -> str:
    """Human-readable module class name."""
    return type(module).__name__


def flatten_weight_rows(W: torch.Tensor, module: nn.Module) -> torch.Tensor:
    """Reshape any weight tensor to [out_channels, flat_in] for per-row quant."""
    W = W.float()
    if isinstance(module, (nn.Conv1d, nn.Conv2d, nn.Conv3d)):
        return W.reshape(W.shape[0], -1)  # each filter = one row
    return W  # Linear / Embedding already [out, in]


def layer_num_params(module: nn.Module) -> int:
    """Count weight + bias parameters for any quantizable module."""
    if hasattr(module, "weight") and module.weight is not None:
        n = module.weight.numel()
        if getattr(module, "bias", None) is not None:
            n += module.bias.numel()
        return n
    return sum(p.numel() for p in module.parameters(recurse=False))


def symmetric_dequant_weight(state: WeightQuantState) -> torch.Tensor:
    """Dequantize generic WeightQuantState back to float weights."""
    rows = symmetric_dequant_per_channel(
        state.weight_q.reshape(state.weight_shape[0], -1), state.weight_scales)
    return rows.reshape(state.weight_shape)


# ═══════════════════════════════════════════════════════════════
# Quantized Conv1d — per-output-channel dequant + conv
# ═══════════════════════════════════════════════════════════════
class QuantizedConv1d(nn.Module):
    """Drop-in Conv1d with per-output-channel int quant weights."""

    def __init__(self, conv: nn.Conv1d, state: WeightQuantState):
        super().__init__()
        self.stride, self.padding, self.dilation = conv.stride, conv.padding, conv.dilation
        self.groups = conv.groups
        self.in_channels, self.out_channels = conv.in_channels, conv.out_channels
        self.kernel_size = conv.kernel_size
        self.method, self.n_bits = state.method, state.n_bits
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None

    @property
    def weight_fp(self) -> torch.Tensor:
        return symmetric_dequant_weight(WeightQuantState(
            self.weight_q, self.weight_scales, tuple(self.weight_q.shape), "Conv1d"))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.weight_fp.to(x.dtype)
        return F.conv1d(x, w, self.bias, self.stride, self.padding, self.dilation, self.groups)

    def storage_bytes(self) -> int:
        n = self.weight_q.numel() + self.weight_scales.numel()
        if self.bias is not None:
            n += self.bias.numel()
        return n * 4


# ═══════════════════════════════════════════════════════════════
# Quantized Conv2d — per-output-channel dequant + conv (ViT/CNN)
# ═══════════════════════════════════════════════════════════════
class QuantizedConv2d(nn.Module):
    """Drop-in Conv2d with per-output-channel int quant weights (ViT/CNN vision blocks)."""

    def __init__(self, conv: nn.Conv2d, state: WeightQuantState):
        super().__init__()
        self.stride, self.padding = conv.stride, conv.padding
        self.dilation, self.groups = conv.dilation, conv.groups
        self.in_channels, self.out_channels = conv.in_channels, conv.out_channels
        self.kernel_size = conv.kernel_size
        self.method, self.n_bits = state.method, state.n_bits
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None

    @property
    def weight_fp(self) -> torch.Tensor:
        return symmetric_dequant_weight(WeightQuantState(
            self.weight_q, self.weight_scales, tuple(self.weight_q.shape), "Conv2d"))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.weight_fp.to(x.dtype)
        return F.conv2d(x, w, self.bias, self.stride, self.padding, self.dilation, self.groups)

    def storage_bytes(self) -> int:
        n = self.weight_q.numel() + self.weight_scales.numel()
        if self.bias is not None:
            n += self.bias.numel()
        return n * 4


# ═══════════════════════════════════════════════════════════════
# Quantized Conv3d — per-output-channel dequant + conv
# ═══════════════════════════════════════════════════════════════
class QuantizedConv3d(nn.Module):
    """Drop-in Conv3d with per-output-channel int quant weights."""

    def __init__(self, conv: nn.Conv3d, state: WeightQuantState):
        super().__init__()
        self.stride, self.padding, self.dilation = conv.stride, conv.padding, conv.dilation
        self.groups = conv.groups
        self.in_channels, self.out_channels = conv.in_channels, conv.out_channels
        self.kernel_size = conv.kernel_size
        self.method, self.n_bits = state.method, state.n_bits
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None

    @property
    def weight_fp(self) -> torch.Tensor:
        return symmetric_dequant_weight(WeightQuantState(
            self.weight_q, self.weight_scales, tuple(self.weight_q.shape), "Conv3d"))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.weight_fp.to(x.dtype)
        return F.conv3d(x, w, self.bias, self.stride, self.padding, self.dilation, self.groups)

    def storage_bytes(self) -> int:
        n = self.weight_q.numel() + self.weight_scales.numel()
        if self.bias is not None:
            n += self.bias.numel()
        return n * 4


# ═══════════════════════════════════════════════════════════════
# Quantized Embedding — per-row dequant + lookup
# ═══════════════════════════════════════════════════════════════
class QuantizedEmbedding(nn.Module):
    """Drop-in Embedding with per-row int quant lookup table."""

    def __init__(self, emb: nn.Embedding, state: WeightQuantState):
        super().__init__()
        self.num_embeddings, self.embedding_dim = emb.num_embeddings, emb.embedding_dim
        self.padding_idx, self.max_norm = emb.padding_idx, emb.max_norm
        self.norm_type, self.scale_grad_by_freq = emb.norm_type, emb.scale_grad_by_freq
        self.sparse = emb.sparse
        self.method, self.n_bits = state.method, state.n_bits
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)

    @property
    def weight_fp(self) -> torch.Tensor:
        return symmetric_dequant_weight(WeightQuantState(
            self.weight_q, self.weight_scales, tuple(self.weight_q.shape), "Embedding"))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.weight_fp.to(torch.float32)
        return F.embedding(x, w, self.padding_idx, self.max_norm, self.norm_type,
                           self.scale_grad_by_freq, self.sparse)

    def storage_bytes(self) -> int:
        return (self.weight_q.numel() + self.weight_scales.numel()) * 4


# ═══════════════════════════════════════════════════════════════
# Generic RTN — round-to-nearest for Linear / Conv / Embedding
# ═══════════════════════════════════════════════════════════════
class GenericRTNQuantizer:
    """Round-to-nearest per-output-channel quant — works on Linear/Conv/Embedding."""

    def __init__(self, module: nn.Module, n_bits: int = 4):
        self.module = module
        self.n_bits = n_bits
        self.calib = []

    def add_batch(self, inp: torch.Tensor):
        self.calib.append(inp.detach())

    def quantize(self) -> WeightQuantState:
        W = self.module.weight.data
        shape = tuple(W.shape)
        W2 = flatten_weight_rows(W, self.module)
        q, s = symmetric_quantize_per_channel(W2, self.n_bits)
        bias = self.module.bias.data.clone() if getattr(self.module, "bias", None) is not None else None
        return WeightQuantState(q.reshape(shape), s, shape, module_type_name(self.module), bias, self.n_bits, "rtn")


# ═══════════════════════════════════════════════════════════════
# Generic AWQ — activation-aware scales for Conv* (Embedding → RTN)
# ═══════════════════════════════════════════════════════════════
class GenericAWQQuantizer:
    """Activation-aware grid search on flattened weights — Conv*; Embedding falls back to RTN."""

    def __init__(self, module: nn.Module, n_bits: int = 4, grid_steps: int = 20):
        self.module = module
        self.n_bits = n_bits
        self.grid_steps = grid_steps
        self.calib = []

    def add_batch(self, inp: torch.Tensor):
        self.calib.append(inp.detach())

    def quantize(self) -> WeightQuantState:
        if isinstance(self.module, nn.Embedding):
            return GenericRTNQuantizer(self.module, self.n_bits).quantize()
        W = self.module.weight.data.float()
        shape = tuple(W.shape)
        W2 = flatten_weight_rows(W, self.module)
        act_scales = torch.ones(W2.shape[1], device=W.device, dtype=W.dtype)
        for x in self.calib:
            if isinstance(self.module, nn.Conv1d):
                ch_max = x.abs().amax(dim=(0, 2), keepdim=False).flatten()
            elif isinstance(self.module, nn.Conv2d):
                ch_max = x.abs().amax(dim=(0, 2, 3), keepdim=False).flatten()
            elif isinstance(self.module, nn.Conv3d):
                ch_max = x.abs().amax(dim=tuple(range(1, x.dim())), keepdim=False).flatten()
            else:
                ch_max = x.reshape(-1, x.shape[-1]).abs().amax(dim=0)
            n = min(ch_max.numel(), act_scales.numel())
            act_scales[:n] = torch.maximum(act_scales[:n], ch_max[:n].to(act_scales.device))
        best_q, best_s, best_err = None, None, float("inf")
        for step in range(self.grid_steps + 1):
            ratio = step / max(self.grid_steps, 1)
            s_act = act_scales.pow(ratio).clamp(min=1e-8)
            Ws = W2 / s_act.unsqueeze(0)
            q, sc = symmetric_quantize_per_channel(Ws, self.n_bits)
            W_hat = symmetric_dequant_per_channel(q, sc) * s_act.unsqueeze(0)
            err = (W2 - W_hat).pow(2).mean().item()
            if err < best_err:
                best_err, best_q, best_s = err, q, sc
        bias = self.module.bias.data.clone() if getattr(self.module, "bias", None) is not None else None
        return WeightQuantState(best_q.reshape(shape), best_s, shape, module_type_name(self.module), bias, self.n_bits, "awq")



# ═══════════════════════════════════════════════════════════════
# WeightQuant Linear — RTN/AWQ fallback drop-in
# ═══════════════════════════════════════════════════════════════
class WeightQuantLinear(nn.Module):
    """Drop-in Linear with generic per-channel WeightQuantState (RTN/AWQ fallback)."""

    def __init__(self, linear: nn.Linear, state: WeightQuantState):
        super().__init__()
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.method, self.n_bits = state.method, state.n_bits
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None

    @property
    def weight_fp(self) -> torch.Tensor:
        return symmetric_dequant_weight(WeightQuantState(
            self.weight_q, self.weight_scales, tuple(self.weight_q.shape), "Linear"))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.weight_fp.to(x.dtype)
        return F.linear(x, w, self.bias)

    def storage_bytes(self) -> int:
        n = self.weight_q.numel() + self.weight_scales.numel()
        if self.bias is not None:
            n += self.bias.numel()
        return n * 4


# ═══════════════════════════════════════════════════════════════
# Factory: build the right quantized module from any layer + QuantState
# ═══════════════════════════════════════════════════════════════
def build_quantized_module(orig: nn.Module, state) -> nn.Module:
    """Build the correct quantized drop-in module for any supported layer type."""
    if isinstance(orig, nn.Linear) and not isinstance(state, WeightQuantState):
        return QuantizedLinear(orig.in_features, orig.out_features, state)
    if isinstance(state, WeightQuantState):
        if isinstance(orig, nn.Conv1d):
            return QuantizedConv1d(orig, state)
        if isinstance(orig, nn.Conv2d):
            return QuantizedConv2d(orig, state)
        if isinstance(orig, nn.Conv3d):
            return QuantizedConv3d(orig, state)
        if isinstance(orig, nn.Embedding):
            return QuantizedEmbedding(orig, state)
        if isinstance(orig, nn.Linear):
            return WeightQuantLinear(orig, state)
    raise ValueError(f"Unsupported pair: {type(orig).__name__} + {type(state).__name__}")


print("Building blocks loaded — Linear/Conv/Embedding support for LLM, VLM, ViT, CNN.")


# ═══════════════════════════════════════════════════════════════
# Helpers: original forward + weight/output MSE
# ═══════════════════════════════════════════════════════════════
def _forward_orig(module: nn.Module, x: torch.Tensor) -> torch.Tensor:
    """Run original module forward (float weights) for any quantizable type."""
    if isinstance(module, nn.Linear):
        return F.linear(x, module.weight.float(), module.bias)
    if isinstance(module, nn.Embedding):
        return module(x.long())
    return module(x)


def layer_weight_mse(orig: nn.Module, quant: nn.Module) -> float:
    """Mean squared error between original float weight and reconstructed weight."""
    with torch.no_grad():
        if not hasattr(quant, "weight_fp"):
            return float("nan")
        err = (orig.weight.float() - quant.weight_fp).pow(2).mean().item()
    return err


def layer_output_mse(orig: nn.Module, quant: nn.Module, inputs: torch.Tensor | None) -> float:
    """Output MSE: measures how much quantization distorts the layer's actual output."""
    if inputs is None or inputs.numel() == 0:
        return float("nan")
    if isinstance(orig, nn.Embedding):
        x = inputs[:2048].long().to(orig.weight.device)
    else:
        x = inputs[:2048].to(orig.weight.device)
    with torch.no_grad():
        y_fp = _forward_orig(orig, x)
        y_q = quant(x)
        return (y_fp.float() - y_q.float()).pow(2).mean().item()


---
## Stage 2 — GPTQ

📎 **[GPTQState + GPTQLinear — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_gptq_state_linear_lines.png)**



![Stage 2 — GPTQ Hessian-aware column-wise quantization](attachment:nb02_stage02_gptq.png)

Paper: *GPTQ*

📎 **[GPTQQuantizer — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_gptq_quantizer_lines.png)**

📎 **[GPTQLinear — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_gptq_linear_lines.png)**

 (Frantar et al., 2023).

### Objective — step-by-step proof

**Setup (PyTorch `nn.Linear` convention):**

| Symbol | Shape | Meaning |
|--------|-------|---------|
| $\mathbf{X}$ | $M \times I$ | calibration inputs (flattened batch × seq) |
| $\mathbf{W}$ | $O \times I$ | full-precision weight (`layer.weight`) |
| $\hat{\mathbf{W}}$ | $O \times I$ | quantized weight |
| $\mathbf{Y}$ | $M \times O$ | layer output: $\mathbf{Y} = \mathbf{X}\mathbf{W}^\top$ |

GPTQ minimizes the **output reconstruction error** (not weight MSE directly):

$$
\mathcal{L}(\hat{\mathbf{W}}) = \|\mathbf{X}\mathbf{W}^\top - \mathbf{X}\hat{\mathbf{W}}^\top\|_F^2
$$

---

**Step 1 — Introduce the weight error**

Define $\Delta\mathbf{W} = \mathbf{W} - \hat{\mathbf{W}} \in \mathbb{R}^{O \times I}$.

Factor the output difference:

$$
\mathbf{X}\mathbf{W}^\top - \mathbf{X}\hat{\mathbf{W}}^\top
= \mathbf{X}(\mathbf{W} - \hat{\mathbf{W}})^\top
= \mathbf{X}\,\Delta\mathbf{W}^\top
$$

So the loss becomes:

$$
\mathcal{L} = \|\mathbf{X}\,\Delta\mathbf{W}^\top\|_F^2
$$

---

**Step 2 — Frobenius norm as a trace**

For any matrix $\mathbf{A}$, $\|\mathbf{A}\|_F^2 = \text{tr}(\mathbf{A}^\top\mathbf{A})$.

Let $\mathbf{A} = \mathbf{X}\,\Delta\mathbf{W}^\top$ (shape $M \times O$):

$$
\mathcal{L}
= \text{tr}\!\bigl((\mathbf{X}\,\Delta\mathbf{W}^\top)^\top (\mathbf{X}\,\Delta\mathbf{W}^\top)\bigr)
$$

---

**Step 3 — Transpose inside the trace**

Use $(\mathbf{AB})^\top = \mathbf{B}^\top\mathbf{A}^\top$:

$$
(\mathbf{X}\,\Delta\mathbf{W}^\top)^\top = \Delta\mathbf{W}\,\mathbf{X}^\top
$$

Substitute:

$$
\mathcal{L}
= \text{tr}\!\bigl(\Delta\mathbf{W}\,\mathbf{X}^\top\mathbf{X}\,\Delta\mathbf{W}^\top\bigr)
$$

---

**Step 4 — Define the Hessian**

Define the (uncentered) **Hessian** (input Gram matrix):

$$
\mathbf{H} = \mathbf{X}^\top\mathbf{X} \in \mathbb{R}^{I \times I}
$$

$\mathbf{H}$ is symmetric and positive semi-definite (PSD), because $\mathbf{v}^\top\mathbf{H}\mathbf{v} = \|\mathbf{X}\mathbf{v}\|_2^2 \ge 0$ for any $\mathbf{v}$.

Substitute into Step 3:

$$
\mathcal{L}
= \text{tr}\!\bigl((\mathbf{W}-\hat{\mathbf{W}})\,\mathbf{H}\,(\mathbf{W}-\hat{\mathbf{W}})^\top\bigr)
$$

**Result:**

$$
\boxed{
\mathcal{L}(\mathbf{W})
= \|\mathbf{X}\mathbf{W}^\top - \mathbf{X}\hat{\mathbf{W}}^\top\|_F^2
= \text{tr}\!\bigl((\mathbf{W}-\hat{\mathbf{W}})\,\mathbf{H}\,(\mathbf{W}-\hat{\mathbf{W}})^\top\bigr),
\quad \mathbf{H} = \mathbf{X}^\top\mathbf{X}
}
$$

> **Note:** Frantar et al. sometimes write $\mathbf{H} = 2\mathbf{X}^\top\mathbf{X}$ (Optimal Brain Surgeon convention). Our `add_batch` accumulates $\mathbf{H} \mathrel{+}= \mathbf{X}^\top\mathbf{X}$ — same formula, just a constant factor absorbed into the column-update scaling.

---

**Step 5 — Why this is a quadratic form (one output channel view)**

Let $\Delta\mathbf{w}_i \in \mathbb{R}^{I}$ be row $i$ of $\Delta\mathbf{W}$ (output channel $i$). Then:

$$
\mathcal{L} = \sum_{i=1}^{O} \Delta\mathbf{w}_i^\top \mathbf{H}\, \Delta\mathbf{w}_i
= \sum_{i=1}^{O} \Delta\mathbf{w}_i^\top \mathbf{X}^\top\mathbf{X}\, \Delta\mathbf{w}_i
$$

Each term is a **weighted squared error** — directions of $\mathbf{X}$ with large variance (large $\mathbf{H}_{jj}$) are penalized more. That is why GPTQ quantizes columns in Hessian order and compensates remaining columns using $\mathbf{H}^{-1}$.

### Column-by-column update (how GPTQ uses H)

After quantizing column $i$ to $\hat{w}_i$, the error $\delta_i = w_i - \hat{w}_i$ is **spread to unquantized columns** $j > i$ using $\mathbf{H}^{-1}$:

$$
\Delta w_j = -\frac{\delta_i}{\mathbf{H}_{ii}} \mathbf{H}_{ij}, \quad j > i
$$

**In code:** `GPTQQuantizer.quantize()` precomputes $\mathbf{H}^{-1}$ via Cholesky, then loops columns left-to-right — quantize column $i$, compensate columns $j > i$, repeat.

**Why it works:** high-variance input directions (large $\mathbf{H}_{ii}$) get quantized first; remaining columns absorb the error so output $\mathbf{X}\mathbf{W}^\top$ stays close to fp16.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# GPTQ — Hessian-aware column-by-column quantization
# ═══════════════════════════════════════════════════════════════
class GPTQQuantizer:
    """GPTQ for one nn.Linear — Hessian-aware column-by-column quantization."""

    def __init__(
        self,
        layer: nn.Linear,
        n_bits: int = 4,
        block_size: int = 128,
        damping: float = 0.01,
    ):
        self.layer = layer
        self.n_bits = n_bits
        self.block_size = block_size  # process columns in blocks for efficiency
        self.damping = damping        # regularization on Hessian diagonal
        self.H: torch.Tensor | None = None  # accumulated Hessian (X^T @ X)
        self.nsamples = 0

    def add_batch(self, inp: torch.Tensor):
        """Accumulate Hessian from calibration inputs: H += X^T @ X."""
        if inp.dim() == 3:
            inp = inp.reshape(-1, inp.shape[-1])  # flatten batch+seq → rows
        inp = inp.float()
        if self.H is None:
            self.H = torch.zeros((inp.shape[1], inp.shape[1]), device=inp.device)
        batch = inp.shape[0]
        self.H += inp.t() @ inp  # outer product accumulation
        self.nsamples += batch

    def quantize(self) -> GPTQState:
        W = self.layer.weight.data.float().clone()  # working copy of weights
        H = self.H.clone()
        dead = torch.diag(H) == 0  # columns with zero variance (unused inputs)
        H[dead, dead] = 1.0        # avoid singularity for dead columns
        W[:, dead] = 0.0           # zero out weights for dead inputs

        damp = self.damping * torch.mean(torch.diag(H))  # adaptive damping
        diag_idx = torch.arange(H.shape[0], device=H.device)
        H[diag_idx, diag_idx] += damp  # regularize diagonal for numerical stability

        H = torch.linalg.cholesky(H)          # Cholesky factorization of Hessian
        Hinv = torch.cholesky_inverse(H)      # H^{-1} via Cholesky
        Hinv = torch.linalg.cholesky(Hinv, upper=True)  # upper-triangular for column updates

        Q = torch.zeros_like(W)  # quantized weight accumulator
        qmax = qmax_for_bits(self.n_bits)

        for i1 in range(0, W.shape[1], self.block_size):  # process in column blocks
            i2 = min(i1 + self.block_size, W.shape[1])
            count = i2 - i1
            W1 = W[:, i1:i2].clone()       # current block of weights
            Q1 = torch.zeros_like(W1)       # quantized block
            Err1 = torch.zeros_like(W1)     # error residuals
            Hinv1 = Hinv[i1:i2, i1:i2]     # Hessian inverse for this block

            for i in range(count):  # quantize one column at a time
                w = W1[:, i]
                d = Hinv1[i, i]                          # diagonal element (weight importance)
                max_abs = w.abs().max().clamp(min=1e-8)  # column-wise scale
                scale = max_abs / qmax
                q = torch.round(w / scale).clamp(-qmax - 1, qmax)  # quantize this column
                Q1[:, i] = q
                err = (w - q * scale) / d  # quantization error weighted by inverse Hessian
                W1[:, i:] -= err.unsqueeze(1) @ Hinv1[i, i:].unsqueeze(0)  # propagate error to remaining cols
                Err1[:, i] = err

            Q[:, i1:i2] = Q1
            W[:, i2:] -= Err1 @ Hinv[i1:i2, i2:]  # propagate block error to future blocks

        weight_q, weight_scales = symmetric_quantize_per_channel(Q, self.n_bits)  # final per-channel scales
        return GPTQState(
            weight_q=weight_q.cpu(),
            weight_scales=weight_scales.cpu(),
            bias=self.layer.bias.detach().cpu() if self.layer.bias is not None else None,
            n_bits=self.n_bits,
        )


print("GPTQ quantizer ready.")


---
## Stage 3 — AWQ

📎 **[AWQState + AWQLinear — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_awq_state_linear_lines.png)**



![Stage 3 — AWQ activation-aware quantization](attachment:nb02_stage03_awq.png)

Paper: *AWQ*

📎 **[AWQQuantizer — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_awq_quantizer_lines.png)**

📎 **[AWQLinear — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_awq_linear_lines.png)**

 (Lin et al., 2023).

### Weighted output error

For one output element $y = \sum_j x_j w_j$, quantization error:

$$
\Delta y = \sum_j x_j (w_j - \hat{w}_j)
$$

Under independence approximation:

$$
\mathbb{E}[\Delta y^2] \approx \sum_j \mathbb{E}[x_j^2] \cdot \mathbb{E}[(w_j - \hat{w}_j)^2]
$$

**Proof (variance propagation):** if $\text{Cov}(x_j, w_j - \hat{w}_j) = 0$ and channels independent, $\text{Var}(\sum_j x_j \epsilon_j) = \sum_j \mathbb{E}[x_j^2]\mathbb{E}[\epsilon_j^2]$ where $\epsilon_j = w_j - \hat{w}_j$.

### Per-channel scaling

Search $s_j > 0$ with $\hat{w}_j = Q(w_j / s_j) \cdot s_j$. Grid over $r \in [0,1]$:

$$
s_j(r) \propto \left(\max_m |X_j^{(m)}|\right)^r
$$

Pick $r^\star = \arg\min_r \sum_m \|\mathbf{X}^{(m)}\mathbf{W}^\top - \mathbf{X}^{(m)}\hat{\mathbf{W}}(r)^\top\|_2^2$.

**Intuition proof:** large $|x_j|$ inflates $\mathbb{E}[x_j^2]$; increasing $s_j$ shrinks $w_j/s_j$ before rounding → smaller effective output error from channel $j$.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# AWQ — activation-aware per-channel scale search
# ═══════════════════════════════════════════════════════════════
class AWQQuantizer:
    """AWQ for one nn.Linear — find per-channel scales that minimize activation-weighted error."""

    def __init__(self, layer: nn.Linear, n_bits: int = 4, grid_steps: int = 20):
        self.layer = layer
        self.n_bits = n_bits
        self.grid_steps = grid_steps       # number of alpha values to search
        self.act_scales: torch.Tensor | None = None  # running max of activation magnitudes

    def add_batch(self, inp: torch.Tensor):
        """Track per-channel max activation magnitude across calibration batches."""
        if inp.dim() == 3:
            inp = inp.reshape(-1, inp.shape[-1])  # flatten to [tokens, channels]
        batch_scale = inp.abs().amax(dim=0).float()  # max per input channel
        if self.act_scales is None:
            self.act_scales = batch_scale
        else:
            self.act_scales = torch.maximum(self.act_scales, batch_scale)  # element-wise max

    def quantize(self) -> AWQState:
        W = self.layer.weight.data.float()
        act = self.act_scales.to(W.device).clamp(min=1e-6)  # activation magnitudes per channel

        best_error = float("inf")
        best_q, best_scales = None, None

        for step in range(self.grid_steps):  # grid search over alpha ∈ [0, 1)
            ratio = step / self.grid_steps   # alpha = how much to weight activations
            s = act.pow(ratio).clamp(min=1e-6)  # per-channel scale: s_j = act_j^alpha
            W_scaled = W * s.unsqueeze(0)       # scale up important channels before quant
            q, ch_scales = symmetric_quantize_per_channel(W_scaled, self.n_bits)  # quantize scaled weight
            W_hat = symmetric_dequant_per_channel(q, ch_scales) / s.unsqueeze(0)  # undo scale after dequant
            err = ((W - W_hat) * act.unsqueeze(0)).pow(2).sum().item()  # activation-weighted MSE
            if err < best_error:
                best_error = err
                best_q, best_scales = q, ch_scales  # keep best across grid

        return AWQState(
            weight_q=best_q.cpu(),
            weight_scales=best_scales.cpu(),
            bias=self.layer.bias.detach().cpu() if self.layer.bias is not None else None,
            n_bits=self.n_bits,
        )


print("AWQ quantizer ready.")


---
## Stage 4 — SmoothQuant

📎 **[SmoothQuantState act_scales — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_smoothquant_act_scales_lines.png)**



![Stage 4 — SmoothQuant outlier migration](attachment:nb02_stage04_smoothquant.png)

Paper: *SmoothQuant*

📎 **[SmoothQuantQuantizer + SmoothQuantLinear — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_smoothquant_quantizer_linear_lines.png)**
 (Xiao et al., 2023).

### Outlier problem (formal)

Define channel dynamic range $R_j = \max_m |X_j^{(m)}| / \mathbb{E}_m[|X_j^{(m)}|]$. Outlier channels have $R_j \gg 1$, forcing int8 scale $s_j^X \propto R_j$ and wasting precision on other channels.

### Migration theorem

For $\alpha \in [0,1]$, per-channel:

$$
s_j = \frac{R_j^\alpha}{\|W_j\|_\infty^{1-\alpha}}, \quad W'_j = \frac{W_j}{s_j}, \quad X'_j = X_j \cdot s_j
$$

**Theorem (exact equivalence):** for all $j$:

$$
X_j W_j = X'_j W'_j = (X_j s_j)(W_j / s_j)
$$

**Proof:** direct algebra; summing over $j$: $\mathbf{x}^\top \mathbf{w} = \mathbf{x}'^\top \mathbf{w}'$ for any row vector. Matmul $\mathbf{X}\mathbf{W}^\top$ unchanged column-wise.

**Effect on ranges:** $|X'_j|_{\max} \approx R_j^{1-\alpha}$ and $|W'_j|_{\max} \approx \|W_j\|_\infty^\alpha$ — trade activation outliers for weight scale at controllable rate $\alpha$. Quantize $\mathbf{W}'$; fold $s_j$ into runtime.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SmoothQuant — migrate outliers from activations to weights
# ═══════════════════════════════════════════════════════════════
class SmoothQuantQuantizer:
    """SmoothQuant — migrate outliers from activations to weights via per-channel scales."""

    def __init__(self, layer: nn.Linear, n_bits: int = 4, alpha: float = 0.5):
        self.layer = layer
        self.n_bits = n_bits
        self.alpha = alpha            # controls how much outlier burden moves to weights
        self.act_max: torch.Tensor | None = None  # per-channel activation max across calibration

    def add_batch(self, inp: torch.Tensor):
        """Track max activation magnitude per input channel."""
        if inp.dim() == 3:
            inp = inp.reshape(-1, inp.shape[-1])  # flatten batch+seq
        batch_max = inp.abs().amax(dim=0).float()  # max per channel in this batch
        if self.act_max is None:
            self.act_max = batch_max
        else:
            self.act_max = torch.maximum(self.act_max, batch_max)  # running max

    def quantize(self) -> SmoothQuantState:
        W = self.layer.weight.data.float()
        act_max = self.act_max.to(W.device).clamp(min=1e-6)       # activation range per channel
        weight_max = W.abs().amax(dim=0).clamp(min=1e-6)           # weight range per input channel

        # s_j = act_max^alpha / weight_max^(1-alpha) — balances both distributions
        s = act_max.pow(self.alpha) / weight_max.pow(1.0 - self.alpha)
        s = s.clamp(min=1e-6)                  # avoid division by zero
        W_smooth = W / s.unsqueeze(0)          # absorb activation outliers into weights
        q, ch_scales = symmetric_quantize_per_channel(W_smooth, self.n_bits)  # quantize smoothed weights

        return SmoothQuantState(
            weight_q=q.cpu(),
            weight_scales=ch_scales.cpu(),
            bias=self.layer.bias.detach().cpu() if self.layer.bias is not None else None,
            act_scales=s.cpu(),  # store scales — needed at inference to scale activations
            n_bits=self.n_bits,
        )


print("SmoothQuant quantizer ready.")


---
## Stage 5 — SpinQuant

📎 **[SpinQuantState + Quantizer + Linear — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_spinquant_all_lines.png)**


![Stage 5 — SpinQuant learned rotation before quantize](attachment:nb02_stage05_spinquant.png)

Paper: *SpinQuant: LLM Quantization with Learned Rotations* (Liu et al., ICLR 2025).

### Rotation invariance

For orthogonal $\mathbf{R}$ ($\mathbf{R}^\top\mathbf{R} = \mathbf{I}$):

$$
\mathbf{Y} = \mathbf{X}\mathbf{W}^\top = (\mathbf{X}\mathbf{R})(\mathbf{W}\mathbf{R})^\top = \mathbf{X}'(\mathbf{W}')^\top
$$

**Proof:** $\mathbf{X}'(\mathbf{W}')^\top = \mathbf{X}\mathbf{R}\mathbf{R}^\top\mathbf{W}^\top = \mathbf{X}\mathbf{W}^\top$ since $\mathbf{R}\mathbf{R}^\top = \mathbf{I}$.

Quantize $\mathbf{W}' = \mathbf{W}\mathbf{R}$ in rotated space where outliers are spread evenly → lower $\text{MSE}_{\text{out}}$ at 4-bit.

### Learned Givens rotations (from scratch)

SpinQuant learns $K$ plane rotations $G_k$:

$$
\mathbf{R} = G_K \cdots G_1, \quad G_k \text{ rotates coordinates } (i_k, j_k) \text{ by angle } \theta_k
$$

We optimize $\{\theta_k\}$ on calibration data:

$$
\min_{\theta} \|\mathbf{X}\mathbf{W}^\top - Q(\mathbf{X}\mathbf{R}_\theta)\, Q(\mathbf{W}\mathbf{R}_\theta)^\top\|_F^2
$$

Then store $(\theta_k, i_k, j_k)$ — compact vs full $\mathbf{R}$ matrix. Stage 15d demo compares SpinQuant vs round-to-nearest on one layer.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SpinQuant — learn Givens rotations, then quantize
# ═══════════════════════════════════════════════════════════════
class SpinQuantQuantizer:
    """SpinQuant — learn Givens rotations to minimize post-quant output error."""

    def __init__(
        self,
        layer: nn.Linear,
        n_bits: int = 4,
        n_givens: int | None = None,
        refine_steps: int | None = None,
        lr: float | None = None,
    ):
        self.layer = layer
        self.n_bits = n_bits
        self.n_givens = n_givens if n_givens is not None else globals().get("SPINQUANT_N_GIVENS", 48)
        self.refine_steps = refine_steps if refine_steps is not None else globals().get("SPINQUANT_REFINE_STEPS", 20)
        self.lr = lr if lr is not None else globals().get("SPINQUANT_LR", 0.05)
        self.calib: list[torch.Tensor] = []  # store calibration inputs for rotation learning

    def add_batch(self, inp: torch.Tensor):
        """Collect calibration data for rotation optimization."""
        if inp.dim() == 3:
            inp = inp.reshape(-1, inp.shape[-1])  # flatten batch+seq
        self.calib.append(inp.detach())

    def _sample_pairs(self, n: int, count: int, device) -> torch.Tensor:
        """Randomly select (i,j) plane pairs for Givens rotations."""
        pairs = []
        for _ in range(count):
            i = torch.randint(0, n, (1,)).item()
            j = torch.randint(0, n, (1,)).item()
            if i == j:
                j = (j + 1) % n  # ensure distinct indices
            pairs.append([i, j])
        return torch.tensor(pairs, device=device, dtype=torch.long)

    def _learn_rotation(self, W: torch.Tensor, X: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Optimize rotation angles to minimize ||y_fp - y_quant||^2 on calibration data."""
        _, I = W.shape
        device = W.device
        max_dim = globals().get("SPINQUANT_MAX_DIM", 1024)
        ng = max(8, self.n_givens // 4) if I > max_dim else self.n_givens  # fewer rotations for large layers
        pairs = self._sample_pairs(I, ng, device)             # random rotation planes
        angles = torch.zeros(ng, device=device, requires_grad=True)  # start at identity (angle=0)
        if X is None or X.numel() == 0:
            return angles.detach().cpu(), pairs.cpu()

        X = X[:2048].to(device).float()  # cap calibration tokens
        bias = self.layer.bias
        opt = torch.optim.Adam([angles], lr=self.lr)  # optimize angles with Adam

        for _ in range(self.refine_steps):
            opt.zero_grad()
            R = givens_rotation_matrix(angles, pairs, I, device)  # build rotation from angles
            W_r = W @ R                                           # rotate weight columns
            q, scales = symmetric_quantize_per_channel(W_r, self.n_bits)  # quantize rotated weight
            W_hat = symmetric_dequant_per_channel(q, scales)      # dequant for error computation
            y_fp = F.linear(X, W, bias)              # ground truth output
            y_q = F.linear(X @ R, W_hat, bias)      # quantized output (with rotated input)
            loss = (y_fp - y_q).pow(2).mean()        # minimize output MSE
            loss.backward()                          # backprop through quant (STE-like)
            opt.step()

        return angles.detach().cpu(), pairs.cpu()

    def quantize(self) -> SpinQuantState:
        W = self.layer.weight.data.float()
        X = torch.cat(self.calib, dim=0) if self.calib else None  # merge all calibration batches
        angles, pairs = self._learn_rotation(W, X)  # learn optimal rotation
        device = W.device
        R = givens_rotation_matrix(angles.to(device), pairs.to(device), W.shape[1], device)
        W_r = W @ R  # apply learned rotation to weights
        q, scales = symmetric_quantize_per_channel(W_r, self.n_bits)  # quantize rotated weights

        return SpinQuantState(
            weight_q=q.cpu(),
            weight_scales=scales.cpu(),
            bias=self.layer.bias.detach().cpu() if self.layer.bias is not None else None,
            givens_angles=angles,   # store angles for inference rotation
            givens_pairs=pairs,     # store pairs for inference rotation
            n_bits=self.n_bits,
        )


print("SpinQuant quantizer ready.")


---
## Stage 6 — ConvRot

📎 **[ConvRotState + Quantizer + Linear — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_convrot_all_lines.png)**


![Stage 6 — ConvRot group-wise Regular Hadamard Transform](attachment:nb02_stage06_convrot.png)

Paper: *ConvRot: Rotation-Based Plug-and-Play 4-bit Quantization* (Huang et al., arXiv 2512.03673).

### Group-wise Regular Hadamard Transform (RHT)

Partition input dimension $K$ into blocks of size $N_0$ (power of 4: 16, 64, 256, 1024):

$$
\mathbf{X} = [\mathbf{X}_1, \ldots, \mathbf{X}_B], \quad \mathbf{W} = [\mathbf{W}_1, \ldots, \mathbf{W}_B], \quad B = \lceil K / N_0 \rceil
$$

Per block, apply regular Hadamard rotation:

$$
\mathbf{Y} = \sum_{i=1}^{B} \text{RHT}(\mathbf{X}_i)\, \text{RHT}(\mathbf{W}_i)^\top
$$

### Regular Hadamard base ($n=4$)

$$
\mathbf{H}_4 = \begin{bmatrix} 1&1&1&-1 \\ 1&1&-1&1 \\ 1&-1&1&1 \\ -1&1&1&1 \end{bmatrix}, \quad
\mathbf{H}_{4^{k+1}} = \mathbf{H}_{4^k} \otimes \mathbf{H}_4
$$

Normalize $\mathbf{H}_n / \sqrt{n}$ so $\mathbf{H}_n \mathbf{H}_n^\top = \mathbf{I}_n$ (**orthogonal**).

**Theorem (column discrepancy):** regular $\mathbf{H}_n$ achieves minimal $\|\mathbf{H}_n^\top \mathbf{1}\|_\infty = \sqrt{n}$, preventing row-wise outlier amplification (vs Sylvester Hadamard).

**Complexity:** global rotation $O(K^2)$ → group-wise ConvRot $O(K)$.

Default $N_0 = 256$ (paper recommendation for accuracy/speed trade-off).


In [ ]:
# ═══════════════════════════════════════════════════════════════
# ConvRot — group-wise Regular Hadamard, then quantize
# ═══════════════════════════════════════════════════════════════
class ConvRotQuantizer:
    """ConvRot — group-wise Regular Hadamard rotation then quantize (no training needed)."""

    def __init__(self, layer: nn.Linear, n_bits: int = 4, group_size: int | None = None):
        self.layer = layer
        self.n_bits = n_bits
        self.group_size = group_size if group_size is not None else globals().get("CONVROT_GROUP_SIZE", 256)
        self.calib: list[torch.Tensor] = []  # collected but not used (plug-and-play — no training)

    def add_batch(self, inp: torch.Tensor):
        """Collect calibration data (kept for API consistency, not used for rotation)."""
        if inp.dim() == 3:
            inp = inp.reshape(-1, inp.shape[-1])
        self.calib.append(inp.detach())

    def quantize(self) -> ConvRotState:
        W = self.layer.weight.data.float()
        device = W.device
        gs = self.group_size
        n = gs
        while n % 4 == 0:  # verify group_size is power of 4
            n //= 4
        if n != 1 or gs < 4:
            raise ValueError(f"CONVROT_GROUP_SIZE must be power of 4 (16/64/256/1024), got {gs}")

        R, pad = convrot_rotation_matrix(W.shape[1], gs, device)  # block-diagonal RHT matrix
        W_pad = F.pad(W, (0, pad)) if pad else W    # pad input dim to be divisible by group_size
        W_rot = W_pad @ R                           # rotate weights: smooths outliers per group
        q, scales = symmetric_quantize_per_channel(W_rot, self.n_bits)  # quantize rotated weights

        return ConvRotState(
            weight_q=q.cpu(),
            weight_scales=scales.cpu(),
            bias=self.layer.bias.detach().cpu() if self.layer.bias is not None else None,
            group_size=gs,   # store for inference-time rotation of activations
            pad=pad,         # store for inference-time padding
            n_bits=self.n_bits,
        )


print("ConvRot quantizer ready.")


---
## Stage 7 — Load the model & calibration image

![Stage 7 — Load Florence-2 and calibration page](attachment:nb02_stage07_load_model.png)

This cell loads Florence-2 and the sample document page. **Everything in Phases A–D uses this same image** for calibration and OCR checks.

After this cell you have:

| Object | Role in later phases |
|--------|---------------------|
| `model` | fp16 reference — used in Phases B, C, D (never naive-quantized) |
| `processor` | Tokenizer + image preprocessor |
| `image` | Calibration page for hooks and OCR verify |
| `PROMPT` | `<OCR_WITH_REGION>` for detect task |

📎 **[Why we keep model AND model_naive](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_run_calibration_lines.png)**

**Phase A** copies `model` → `model_naive` before any destructive quant. Do not quantize `model` until Phase D Stage 12.

For **GPTQ**

📎 **[run_calibration + hooks — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_run_calibration_lines.png)**

 attach to **`nn.Linear`** — that is where the Hessian $\mathbf{H}_\ell$ is defined:

$$
\mathbf{H}_\ell = 2 \sum_{m=1}^{M} (\mathbf{X}_\ell^{(m)})^\top \mathbf{X}_\ell^{(m)}
$$

**Why real OCR data?** $\mathbf{H}_\ell$ weights directions by the calibration distribution. Random Gaussian $\mathbf{X}$ mis-estimates what matters → GPTQ minimizes the wrong objective. Document images + task prompts match production activations.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 7 — load Florence-2 + calibration image
# ═══════════════════════════════════════════════════════════════
# Prerequisite: run install cell (0–1) + Stage 1 building blocks first.
# Creates: processor, model (fp16), image, padded, pad_info()
# Phase A copies model → model_naive; keep `model` fp16 until Phase D Stage 12.

ensure_transformers()  # make sure correct transformers version is loaded
from transformers import AutoProcessor, AutoModelForCausalLM

PROMPT = "<OCR_WITH_REGION>" if TASK == "detect" else "<OCR>"  # task-specific prompt token
dtype = torch.float16 if DEVICE == "cuda" else torch.float32   # fp16 on GPU for speed, fp32 on CPU

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)  # tokenizer + image processor
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=dtype,
    attn_implementation="eager",  # avoid flash-attention issues on older GPUs
).to(DEVICE)
model.eval()  # disable dropout / training-only layers

params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_ID}")
print(f"{params/1e6:.1f}M params  ·  dtype={dtype}")


def pad_info(image):
    w, h = image.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), "white")
    pad_x, pad_y = (side - w) // 2, (side - h) // 2
    canvas.paste(image, (pad_x, pad_y))
    return canvas, pad_x, pad_y, w, h


def load_sample_image():
    try:
        url = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
        return Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
    except Exception:
        img = Image.new("RGB", (640, 480), "white")
        ImageDraw.Draw(img).text((20, 20), "Sample document", fill="black")
        return img


image = load_sample_image()
padded, *_ = pad_info(image)
print(f"Using sample page: {image.size[0]}×{image.size[1]} px")

plt.figure(figsize=(6, 4)); plt.imshow(image); plt.title("Document we'll calibrate on"); plt.axis("off"); plt.show()


---
## Phase A — Baseline: naive uniform quantization

📎 **[Phase A — all functions map (8a→8c)](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_a_overview.png)**

📎 **[Phase A — naive int4 everywhere, OCR baseline](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_a_overview.png)**

> **Read this first.** Phase A answers one simple question: *what happens if I quantize everything to int4 with no planning?*

### What you will do (3 steps)

| Step | Cell | Action | Output |
|------|------|--------|--------|
| **8a** | next | Define Phase A `LayerProfile` + helpers | ready to quantize |
| **8b** | below | RTN int4 on **every** layer (copy of model) | `profiles_a` |
| **8c** | below | OCR on fp16 vs naive int4 | `baseline_ocr` scorecard |

### Why we start here

Most beginners jump straight to GPTQ or mixed precision. That hides the baseline. Phase A gives you a **reference line**:

- **Size:** how much compression you get from dumb int4 everywhere
- **Quality:** how much OCR drops when fragile layers are not protected
- **Later:** Phase D should beat this score at similar size

We quantize a **copy** (`model_naive`). The original fp16 `model` stays untouched for Phases B–D.

### Phase A `LayerProfile`

📎 **[Phase A LayerProfile — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_a_layerprofile_lines.png)**

 — one row per quantized layer

![Phase A LayerProfile — what each field means](attachment:nb02_class_layerprofile_a.png)

Each phase defines its **own** `LayerProfile` class (same name, different fields). Phase A only tracks naive RTN results:

| Field | Meaning |
|-------|---------|
| `name` | Layer path, e.g. `language_model.layers.3.mlp.fc1` |
| `bits` | Always `4` in this phase (uniform int4) |
| `weight_mse` | $\|\mathbf{W} - \hat{\mathbf{W}}\|^2$ after RTN |
| `fp_bytes` | Storage before quant (fp16) |
| `q_bytes` | Storage after quant (int4 + scales) |

Saved as **`profiles_a`** — a list of `LayerProfile` rows, one per layer we swapped.

### Pipeline diagram

```
fp16 Florence-2
      │
      ├─ copy → model_naive
      │              │
      │              ▼
      │         int4 ALL layers (RTN, no plan)
      │              │
      │              ▼
      │         profiles_a  (weight MSE, bytes per layer)
      │              │
      └─ fp16 OCR ───┴── naive int4 OCR
                              │
                              ▼
                      baseline_ocr  ← compare in Phase D
```


📎 **[Stage 8a — Phase A LayerProfile fields](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_a_layerprofile_lines.png)**

#### Stage 8a — Phase A helpers (run this first)

Defines Phase A `LayerProfile`, `apply_naive_uniform_quant()`, and `run_florence_detect()`.

**Requires:** Stages 1–7 complete (quant classes + loaded `model` + calibration image)  
**Output:** helpers used in Stages 8b and 8c below

📎 **[run_florence_detect — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_run_florence_detect_lines.png)**

📎 **[Stage 8a — helper functions overview](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_a_overview.png)**

📎 **[iter_quantizable_modules() — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_a_iter_quantizable_modules.png)**

📎 **[replace_module() + layer_matches_patterns() — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_a_replace_module.png)**

📎 **[load_fresh_model() — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_a_load_fresh_model.png)**



In [ ]:
# ═══════════════════════════════════════════════════════════════
# Phase A — LayerProfile (naive uniform quant, one row per layer)
# ═══════════════════════════════════════════════════════════════
@dataclass
class LayerProfile:
    """Phase A LayerProfile — result of uniform RTN quant on one layer."""
    name: str
    bits: int
    weight_mse: float
    fp_bytes: int
    q_bytes: int


def iter_quantizable_modules(root: nn.Module, limit: int | None = None):
    out = []
    for name, module in root.named_modules():
        if isinstance(module, QUANT_MODULE_TYPES):
            out.append((name, module))
    if limit is not None:
        out = out[:limit]
    return out


def iter_linear_modules(root: nn.Module, limit: int | None = None):
    out = [(n, m) for n, m in iter_quantizable_modules(root) if isinstance(m, nn.Linear)]
    if limit is not None:
        out = out[:limit]
    return out


def layer_matches_patterns(name: str, patterns: tuple) -> bool:
    n = name.lower()
    return any(p.lower() in n for p in patterns)


def replace_module(model, name, new_module):
    parent_name, _, child_name = name.rpartition(".")
    parent = model.get_submodule(parent_name) if parent_name else model
    setattr(parent, child_name, new_module)


def apply_naive_uniform_quant(model, n_bits: int = 4) -> list[LayerProfile]:
    modules = iter_quantizable_modules(model, MAX_QUANT_LAYERS)
    profiles: list[LayerProfile] = []
    for i, (name, mod) in enumerate(modules, 1):
        state = GenericRTNQuantizer(mod, n_bits).quantize()
        qlayer = build_quantized_module(mod, state).to(DEVICE)
        w_mse = layer_weight_mse(mod, qlayer)
        replace_module(model, name, qlayer)
        profiles.append(LayerProfile(
            name=name, bits=n_bits, weight_mse=w_mse,
            fp_bytes=layer_num_params(mod) * 2,
            q_bytes=qlayer.storage_bytes(),
        ))
        if i % 20 == 0 or i == len(modules):
            print(f"  naive RTN: {i}/{len(modules)} layers")
    return profiles


def load_fresh_model():
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, trust_remote_code=True, torch_dtype=dtype, attn_implementation="eager",
    ).to(DEVICE)
    m.eval()
    return m


def run_florence_detect(image, processor, model, device, max_new_tokens=512):
    prompt = "<OCR_WITH_REGION>"
    padded, px, py, ow, oh = pad_info(image)
    inputs = processor(text=prompt, images=padded, return_tensors="pt").to(device)
    inputs["pixel_values"] = inputs["pixel_values"].to(dtype=next(model.parameters()).dtype)
    t0 = time.time()
    with torch.no_grad():
        gen = model.generate(
            input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
            max_new_tokens=max_new_tokens, num_beams=1, use_cache=False,
        )
    elapsed = time.time() - t0
    raw = processor.batch_decode(gen, skip_special_tokens=False)[0]
    side = max(ow, oh)
    parsed = processor.post_process_generation(raw, task=prompt, image_size=(side, side))
    region = parsed.get(prompt, {})
    lines = []
    for quad, label in zip(region.get("quad_boxes", []), region.get("labels", [])):
        xs, ys = quad[0::2], quad[1::2]
        bbox = [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]
        x1 = max(0, min(ow, bbox[0] - px)); y1 = max(0, min(oh, bbox[1] - py))
        x2 = max(0, min(ow, bbox[2] - px)); y2 = max(0, min(oh, bbox[3] - py))
        if x2 > x1 and y2 > y1:
            text = re.sub(r"</?[a-zA-Z_][^>]*>", "", str(label)).strip()
            lines.append({"text": text, "bbox": [x1, y1, x2, y2]})
    return lines, elapsed


print("Phase A LayerProfile — fields: name, bits, weight_mse, fp_bytes, q_bytes")


📎 **[Stage 8b — why naive int4 everywhere?](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_apply_naive_uniform_lines.png)**

### Stage 8b — Naive int4 on every layer

**Run the cell below** after the helpers cell. It:

1. Loads a **fresh fp16 copy** of Florence-2 → `model_naive`
2. Calls `apply_naive_uniform_quant(model_naive, n_bits=4)`
3. For each Linear / Conv / Embedding layer: RTN quantize → swap into graph
4. Builds **`profiles_a`**: list of Phase A `LayerProfile` rows

**What is RTN (round-to-nearest)?**

📎 **[apply_naive_uniform_quant — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_apply_naive_uniform_lines.png)** For each channel (or row), find scale $s = \max|\mathbf{w}| / 7$, round $\mathbf{w}/s$ to integers in $[-8,7]$, store int4 + $s$. No Hessian, no sensitivity — the fastest and crudest method.

**What to look for in the output:**

| Print line | Good sign | Bad sign |
|------------|-----------|----------|
| `Layers quantized: N` | N matches total quantizable layers | N = 0 (check GPU / model load) |
| `Average weight MSE` | $10^{-4}$ – $10^{-2}$ typical | $> 1$ (something broke) |
| `Compression: X×` | X ≈ 3–4× on quant layers | X < 2× (scales dominate) |

**Important:** We do **not** protect

![Stage 8b — what naive quant looks like](attachment:nb02_stage8b_naive.png)
 `lm_head` or embeddings here — that is intentional. This is the worst-case baseline Phase D will improve on.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 8b — naive int4 → profiles_a (Phase A LayerProfile)
# ═══════════════════════════════════════════════════════════════
print("Loading fp16 copy for naive baseline...")
model_naive = load_fresh_model()
fp16_params = sum(p.numel() for p in model_naive.parameters())
print(f"fp16 model: {fp16_params/1e6:.1f}M params\n")

t0 = time.time()
profiles_a: list[LayerProfile] = apply_naive_uniform_quant(model_naive, n_bits=4)
naive_elapsed = time.time() - t0

if profiles_a:
    avg_mse = sum(p.weight_mse for p in profiles_a) / len(profiles_a)
    total_fp = sum(p.fp_bytes for p in profiles_a)
    total_q = sum(p.q_bytes for p in profiles_a)
    print(f"\nNaive int4 done in {naive_elapsed:.1f}s")
    print(f"Layers quantized: {len(profiles_a)}")
    print(f"Average weight MSE: {avg_mse:.2e}")
    print(f"Storage: {total_q/1024/1024:.2f} MB quant  vs  {total_fp/1024/1024:.2f} MB fp16")
    print(f"Compression: {total_fp/max(total_q,1):.2f}×")
else:
    print("No layers quantized.")


📎 **[Stage 8c — why measure OCR baseline?](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_stage08c_baseline_gap.png)**

### Stage 8c — OCR baseline (fp16 vs naive int4)

**Run the cell below** to measure task quality, not weight MSE. OCR cares about **detected text lines**, not how close $\hat{\mathbf{W}}$ is to $\mathbf{W}$.

**Steps inside the cell:**

1. Load another fresh fp16 model → run `<OCR_WITH_REGION>` detect → count lines → `fp16_lines`
2. Run the same detect on `model_naive` (naive int4 from 8b) → `naive_lines`
3. Save **`baseline_ocr`** dict for Phase D Stage 14

**How to read the table:**

📎 **[Stage 8c — OCR baseline gap (fp16 vs naive)](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_stage08c_baseline_gap.png)**



| Column | Meaning |
|--------|---------|
| `Lines` | Number of text regions detected on the calibration page |
| `Infer(s)` | Wall-clock generate time (rough; not the main metric here) |
| `Δ lines` | `naive − fp16` — negative means naive quant **lost** detections |

**Typical beginner outcome:** naive int4 compresses well but **line count drops** (sometimes a lot). That is the motivation for Phases B–D — survey layers, find fragile ones, keep them at fp16/int8.

**Do not skip this cell.**

📎 **[Stage 8c — fp16 vs naive OCR side by side](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_run_florence_detect_lines.png)**
 If you jump to Phase D without `baseline_ocr`, you cannot prove smart quant helped.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 8c — OCR baseline: fp16 vs naive int4
# ═══════════════════════════════════════════════════════════════
print("Running fp16 OCR (reference)...")
model_fp16_ref = load_fresh_model()
fp16_lines, fp16_infer_s = run_florence_detect(image, processor, model_fp16_ref, DEVICE)
del model_fp16_ref
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("Running naive int4 OCR...")
naive_lines, naive_infer_s = run_florence_detect(image, processor, model_naive, DEVICE)

print(f"\n{'Approach':<16} {'Lines':>6} {'Infer(s)':>10}")
print("-" * 36)
print(f"{'fp16':<16} {len(fp16_lines):>6} {fp16_infer_s:>10.2f}")
print(f"{'naive int4':<16} {len(naive_lines):>6} {naive_infer_s:>10.2f}")
print(f"\nΔ lines (naive − fp16): {len(naive_lines) - len(fp16_lines):+d}")

# Keep for Phase D comparison
baseline_ocr = {"fp16_lines": len(fp16_lines), "naive_lines": len(naive_lines),
                "fp16_infer_s": fp16_infer_s, "naive_infer_s": naive_infer_s}


---
## Phase B — Layer survey (who is big?)

📎 **[Phase B — all functions map (9a→9b)](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_b_overview.png)**

📎 **[Phase B — layer inventory and parameter survey](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_b_overview.png)**

> **Question this phase answers:** *Which layers exist, how large are they, and which should we never quantize?*

Phase A already showed that blind int4 hurts OCR. Before we quantize smarter, we **map the model** — like reading a building blueprint before renovation.

### What you will do (2 steps)

| Step | Cell | Action | Output |
|------|------|--------|--------|
| **9a** | next | Define Phase B `LayerProfile` + `build_layer_inventory()` | helpers ready |
| **9b** | below | Survey module types + per-layer table | `profiles_b` |

### Phase B `LayerProfile` — inventory fields

📎 **[Phase B LayerProfile — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_b_layerprofile_lines.png)**

📎 **[Phase B LayerProfile — inventory fields visual](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_b_layerprofile_lines.png)**

| Field | Meaning |
|-------|---------|
| `name` | Full module path in the model |
| `module_type` | `Linear`, `Conv2d`, `Embedding`, … |
| `shape` | Weight tensor shape |
| `num_params` | Parameter count (weights + bias) |
| `pct_of_quantizable` | % of all quantizable params this layer holds |
| `protected` | `True` if name matches `lm_head` / `embed` / `vision` patterns |
| `tiny` | `True` if layer has fewer than `MIN_PARAMS_TO_QUANT` params |

Saved as **`profiles_b`**. Phase C will read `profiles_b` and build a **new** `LayerProfile` (different fields) — they are not the same class at runtime.

### Stage 9 — Which layers should we quantize?

A VLM like Florence-2 is **not** only `nn.Linear` — it also has embeddings, layer norms, attention, and possibly Conv2d in the vision encoder.

#### Full model vs what this notebook quantizes

| Module type | Typical role | % of params | Quantize? | Why |
|-------------|--------------|-------------|-----------|-----|
| **`nn.Linear`** | Q/K/V, MLP, lm_head | **~85–95%** | **Yes** | Dominates size; GPTQ/AWQ/etc. target matmul |
| **`nn.Conv2d`** | Vision patch embed | ~3–10% | **Yes — RTN/AWQ** | Per-channel int4/int8 |
| **`nn.Embedding`** | Token embeddings | ~2–5% | **Yes — int8** | Per-row quant; protect lm_head via patterns |
| **`LayerNorm` / `RMSNorm`** | Pre/post norm | <1% | **No — fp16** | Tiny; scale/shift very sensitive |
| **Attention ops** | softmax, matmul | 0 weight params | **No** | Not weight tensors |

#### Param count for one Linear layer

For `nn.Linear` with shape $(O, I)$:

$$N_{\text{params}} = O \cdot I + O \quad \text{(bias included)}$$

int4 saves $\approx 4\times$ vs fp16 per quantizable layer.

#### What to look for in the output

- **Pie chart (module types):** Linear should dominate — that is where compression wins.
- **`profiles_b` table:** Top 5 layers often hold **>50%** of quantizable params — quantizing those matters most.
- **`PROTECT` flag:** `lm_head`, embeddings, vision projection — never forced to int4 in Phase D.

The code cell below prints the full inventory and a bar chart of the largest layers.


📎 **[Stage 9a — Phase B LayerProfile + survey helpers](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_b_layerprofile_lines.png)**

#### Stage 9a — Phase B helpers (run this first)

Defines Phase B `LayerProfile` and `build_layer_inventory()`.

**Requires:** `model` from Stage 7 (fp16, not `model_naive`)  
**Output:** function ready to build `profiles_b` in the next cell

📎 **[build_layer_inventory — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_build_layer_inventory_lines.png)**

📎 **[Stage 9a — build_layer_inventory flow](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_build_layer_inventory_lines.png)**

📎 **[survey_model_layers() — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_b_survey_model_layers.png)**

📎 **[build_layer_inventory() — deep line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_b_build_layer_inventory_deep.png)**



### Stage 9b — Survey all layers

📎 **[Stage 9b — why survey before quantizing](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_b_overview.png)**

**Run the cell below** to build `profiles_b` inventory from the fp16 `model`.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Phase B — LayerProfile (layer survey, one row per quantizable layer)
# ═══════════════════════════════════════════════════════════════
@dataclass
class LayerProfile:
    """Phase B LayerProfile — size and policy flags for one layer."""
    name: str
    module_type: str
    shape: tuple
    num_params: int
    pct_of_quantizable: float
    protected: bool
    tiny: bool


QUANT_POLICY_BY_TYPE = {
    "Linear":    {"quantize": True,  "methods": "GPTQ / AWQ / SmoothQuant / SpinQuant / ConvRot", "why": "LLM/VLM matmul"},
    "Conv1d":    {"quantize": True,  "methods": "RTN / AWQ per-channel", "why": "1D conv stacks"},
    "Conv2d":    {"quantize": True,  "methods": "RTN / AWQ per-channel", "why": "ViT/CNN vision encoder"},
    "Conv3d":    {"quantize": True,  "methods": "RTN / AWQ per-channel", "why": "3D conv"},
    "Embedding": {"quantize": "int8", "methods": "RTN per-row (int8)", "why": "token lookup"},
    "LayerNorm": {"quantize": False, "methods": "keep fp16", "why": "<1% params"},
    "RMSNorm":   {"quantize": False, "methods": "keep fp16", "why": "same as LayerNorm"},
    "BatchNorm2d": {"quantize": False, "methods": "fold into Conv or fp16", "why": "running stats"},
}


def survey_model_layers(model: nn.Module) -> tuple[list[dict], int]:
    from collections import defaultdict
    total = sum(p.numel() for p in model.parameters())
    buckets = defaultdict(lambda: {"count": 0, "params": 0, "example": ""})
    for name, mod in model.named_modules():
        if name == "":
            continue
        n = sum(p.numel() for p in mod.parameters(recurse=False))
        if n == 0:
            continue
        key = type(mod).__name__
        buckets[key]["count"] += 1
        buckets[key]["params"] += n
        if not buckets[key]["example"]:
            buckets[key]["example"] = name
    rows = []
    for typ, info in buckets.items():
        policy = QUANT_POLICY_BY_TYPE.get(typ, {"quantize": False, "methods": "fp16", "why": "keep fp16"})
        rows.append({
            "type": typ, "count": info["count"], "params": info["params"],
            "pct": 100.0 * info["params"] / max(total, 1),
            "example": info["example"], "quantize": policy["quantize"],
            "methods": policy["methods"], "why": policy["why"],
        })
    rows.sort(key=lambda r: r["params"], reverse=True)
    return rows, total


def symmetric_quantize_conv2d(W: torch.Tensor, n_bits: int = 8):
    out_ch = W.shape[0]
    W2 = W.float().reshape(out_ch, -1)
    q, scales = symmetric_quantize_per_channel(W2, n_bits)
    return q.reshape(W.shape), scales


def iter_conv_modules(root: nn.Module):
    return [(n, m) for n, m in root.named_modules() if isinstance(m, nn.Conv2d)]


def build_layer_inventory(all_modules) -> tuple[list[LayerProfile], int]:
    total = sum(layer_num_params(m) for _, m in all_modules)
    profiles: list[LayerProfile] = []
    for name, mod in all_modules:
        n = layer_num_params(mod)
        profiles.append(LayerProfile(
            name=name, module_type=module_type_name(mod),
            shape=tuple(mod.weight.shape), num_params=n,
            pct_of_quantizable=100.0 * n / max(total, 1),
            protected=layer_matches_patterns(name, ALWAYS_FP16_PATTERNS),
            tiny=n < MIN_PARAMS_TO_QUANT,
        ))
    profiles.sort(key=lambda p: p.num_params, reverse=True)
    return profiles, total


print("Phase B LayerProfile — fields: name, shape, num_params, protected, tiny")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 9 — survey ALL module types → profiles_b (Phase B LayerProfile)
# ═══════════════════════════════════════════════════════════════
layer_survey, total_model_params = survey_model_layers(model)
linear_survey = next((r for r in layer_survey if r["type"] == "Linear"), None)
conv_survey = [r for r in layer_survey if r["type"] == "Conv2d"]

print(f"Florence-2 total params: {total_model_params/1e6:.1f}M\n")
print(f"{'Type':<22} {'#Layers':>7} {'Params':>11} {'%':>6}  Quant?   Why")
print("-" * 100)
for row in layer_survey[:12]:
    q = "YES" if row["quantize"] is True else ("int8" if row["quantize"] == "int8" else "fp16")
    print(f"{row['type']:<22} {row['count']:>7} {row['params']:>11,} {row['pct']:>5.1f}%  {q:<8} {row['why'][:40]}")

if linear_survey:
    print(f"\n→ Linear = {linear_survey['pct']:.1f}% of model — Stages 2-6 quantize nn.Linear (biggest win).")
if conv_survey:
    total_conv = sum(r["params"] for r in conv_survey)
    print(f"→ Conv2d = {100*total_conv/max(total_model_params,1):.1f}% — use symmetric_quantize_conv2d() → int8 for mobile (notebook 03).")
    for n, conv in iter_conv_modules(model)[:3]:
        q, s = symmetric_quantize_conv2d(conv.weight.data, 8)
        err = (conv.weight.float() - symmetric_dequant_per_channel(q.reshape(conv.weight.shape[0], -1), s).reshape(conv.weight.shape)).pow(2).mean().item()
        print(f"   demo: {n}  shape={tuple(conv.weight.shape)}  int8 weight MSE={err:.2e}")
else:
    print("→ No standalone Conv2d in this checkpoint (Florence-2 vision may use Linear patch embed). Policy still applies to other VLMs.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
top = layer_survey[:7]
axes[0].pie([r["pct"] for r in top], labels=[r["type"] for r in top], autopct="%1.1f%%", startangle=90)
axes[0].set_title("Params by module type (whole model)")
quant_colors = ["#27ae60" if r["quantize"] is True else ("#f39c12" if r["quantize"] == "int8" else "#e74c3c") for r in top]
axes[1].barh([r["type"] for r in top][::-1], [r["pct"] for r in top][::-1], color=quant_colors[::-1])
axes[1].set_xlabel("% of total params"); axes[1].set_title("Green=quantize Linear, orange=int8 Conv, red=keep fp16")
plt.tight_layout(); plt.show()

print("\n" + "=" * 70)
print("Stage 9 inventory: ALL quantizable layers (Linear + Conv* + Embedding)")
print("=" * 70 + "\n")

all_quantizable = iter_quantizable_modules(model, MAX_ANALYZE_LAYERS or MAX_QUANT_LAYERS)
profiles_b, total_quant_params = build_layer_inventory(all_quantizable)
all_linears = [(n, m) for n, m in all_quantizable if isinstance(m, nn.Linear)]

print(f"Found {len(profiles_b)} quantizable layers  ·  {total_quant_params/1e6:.2f}M params total\n")
print(f"{'Layer':<45} {'Type':<10} {'Shape':<18} {'Params':>9} {'%':>6}  Flag")
print("-" * 105)
for row in profiles_b[:20]:
    flag = "PROTECT" if row.protected else ("tiny" if row.tiny else "quant?")
    short = row.name if len(row.name) <= 44 else "…" + row.name[-43:]
    print(f"{short:<45} {row.module_type:<10} {str(row.shape):<18} {row.num_params:>9,} {row.pct_of_quantizable:>5.1f}%  {flag}")
if len(profiles_b) > 20:
    print(f"... +{len(profiles_b)-20} more layers")

top5_pct = sum(r.pct_of_quantizable for r in profiles_b[:5])
print(f"\nTop 5 layers hold {top5_pct:.0f}% of all quantizable params — quantizing these gives the biggest win.")

fig, ax = plt.subplots(figsize=(10, 4))
top = profiles_b[:15]
ax.barh([r.name.split(".")[-1] for r in top][::-1], [r.pct_of_quantizable for r in top][::-1], color="steelblue")
ax.set_xlabel("% of quantizable params")
ax.set_title("Biggest layers first — Phase B LayerProfile")
plt.tight_layout(); plt.show()


---
## Phase C — Sensitivity ranking (who breaks under int4?)

📎 **[Phase C — all functions map (10a→10b)](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_c_overview.png)**

📎 **[Phase C — sensitivity ranking by output MSE](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_c_overview.png)**

> **Question this phase answers:** *If I quantize layer $\ell$ to int4, how much does its **output** change on real OCR data?*

Weight MSE alone is misleading. A layer can have low weight error but sit on a **high-gradient path** — quantizing it still destroys OCR.

### What you will do (2 steps)

| Step | Cell | Action | Output |
|------|------|--------|--------|
| **10a** | next | Define Phase C `LayerProfile` + calibration helpers | helpers ready |
| **10b** | below | Capture activations → rank layers | `profiles_c` |

### Phase C `LayerProfile` — sensitivity fields

📎 **[Phase C LayerProfile — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_c_layerprofile_lines.png)**

📎 **[Phase C LayerProfile — sensitivity fields visual](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_c_layerprofile_lines.png)**

| Field | Meaning |
|-------|---------|
| `name`, `module_type`, `shape`, `num_params` | Copied from Phase B inventory |
| `protected` | From `profiles_b` |
| `weight_mse_int4`, `weight_mse_int8` | Cheap RTN weight error (no calibration) |
| `output_mse_int4`, `output_mse_int8` | **Real** output shift on calibration batch |
| `act_max` | $A_\ell = \max |X_\ell|$ — activation outlier size |
| `sensitivity` | Ranking score $S_\ell$ (see below) |

Saved as **`profiles_c`**, sorted **highest sensitivity first**.

### Stage 10 — Which layers are sensitive?

#### Layer output error (what we actually measure)

$$
\text{OMSE}_\ell^{(b)} = \frac{1}{M}\sum_m \|\mathbf{Y}^{(m)} - \hat{\mathbf{Y}}^{(m)}\|_F^2, \quad \mathbf{Y}^{(m)} = f_\ell(\mathbf{X}^{(m)})
$$

We run Florence-2 on the calibration image, hook each layer's input $\mathbf{X}_\ell$, quantize weights to $b$ bits, and compare outputs.

#### Sensitivity score (used in Phase D)

$$S_\ell = \text{OMSE}_\ell^{(4)} \cdot \bigl(1 + 0.1 \log(1 + A_\ell)\bigr)$$

- High **OMSE @ int4** → quantizing this layer hurts
- High **act_max** → outlier activations amplify the hurt
- **Higher $S_\ell$** → keep fp16 or int8 in Phase D

#### Error propagation (why early layers matter)

If layer $\ell$ output error is $\Delta \mathbf{Y}$ and the next layer is linear:

$$\|\Delta \mathbf{Z}\|_F \le \|\Delta \mathbf{Y}\|_F \cdot \|\mathbf{W}_{\ell+1}\|_2$$

Errors in early vision / embedding layers **amplify** through the stack. The bar chart below shows who ranks at the top.

#### What to look for in the output

| Chart / table | What it tells you |
|---------------|-------------------|
| Sensitivity bar chart | Red / tall bars = quantize last or use int8 |
| int4 vs int8 output MSE | int8 almost always safer than int4 for same layer |
| `[PROTECT]` tag | Will stay fp16 regardless of rank |


![Stage 10a — Phase C LayerProfile + calibration hooks](attachment:nb02_stage10a_layerprofile.png)

#### Stage 10a — Phase C helpers (run this first)

Defines Phase C `LayerProfile`, calibration hooks, `build_quantizer()`, and `analyze_sensitivity()`.

📎 **[build_quantizer() factory — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_build_quantizer_lines.png)**

**Requires:** `profiles_b` from Phase B (run Phase B cells first)  
**Output:** function ready to build `profiles_c`

📎 **[Stage 10a — calibration hook flow](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_build_quantizer_lines.png)**

📎 **[build_quantizer_for_module() — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_c_build_quantizer_for_module.png)**

📎 **[naive_weight_mse() — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_c_naive_weight_mse.png)**

📎 **[measure_output_mse() — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_c_measure_output_mse.png)**

📎 **[register_hooks + collect_layer_inputs() — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_c_collect_layer_inputs.png)**



### Stage 10b — Run sensitivity analysis

📎 **[analyze_sensitivity() — deep line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_c_analyze_sensitivity_deep.png)**


![Stage 10b — why rank by output MSE, not weight MSE](attachment:nb02_stage10b_why_sensitivity.png)

**Run the cell below** to capture activations and build `profiles_c`.

📎 **[analyze_sensitivity — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_analyze_sensitivity_lines.png)**

📎 **[Stage 10b — sensitivity ranking output](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_c_overview.png)**


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Phase C — LayerProfile (sensitivity ranking, one row per layer)
# ═══════════════════════════════════════════════════════════════
@dataclass
class LayerProfile:
    """Phase C LayerProfile — fragility score from calibration data."""
    name: str
    module_type: str
    shape: tuple
    num_params: int
    protected: bool
    weight_mse_int4: float
    weight_mse_int8: float
    output_mse_int4: float
    output_mse_int8: float
    act_max: float
    sensitivity: float


def build_quantizer(method: str, layer: nn.Linear, n_bits: int | None = None):
    bits = n_bits if n_bits is not None else BITS
    if method == "gptq":
        return GPTQQuantizer(layer, n_bits=bits, block_size=GPTQ_BLOCK_SIZE, damping=GPTQ_DAMPING)
    if method == "awq":
        return AWQQuantizer(layer, n_bits=bits, grid_steps=AWQ_GRID_STEPS)
    if method == "smoothquant":
        return SmoothQuantQuantizer(layer, n_bits=bits, alpha=SMOOTHQUANT_ALPHA)
    if method == "spinquant":
        return SpinQuantQuantizer(layer, n_bits=bits, n_givens=SPINQUANT_N_GIVENS,
                                  refine_steps=SPINQUANT_REFINE_STEPS, lr=SPINQUANT_LR)
    if method == "convrot":
        return ConvRotQuantizer(layer, n_bits=bits, group_size=CONVROT_GROUP_SIZE)
    raise ValueError(f"Unknown QUANT_METHOD: {method}")


def build_quantizer_for_module(method: str, module: nn.Module, n_bits: int | None = None):
    bits = n_bits if n_bits is not None else BITS
    if isinstance(module, nn.Linear):
        return build_quantizer(method, module, bits)
    if method == "awq":
        return GenericAWQQuantizer(module, bits, AWQ_GRID_STEPS)
    if method in LINEAR_ONLY_METHODS and isinstance(module, (nn.Conv1d, nn.Conv2d, nn.Conv3d)):
        return GenericAWQQuantizer(module, bits, AWQ_GRID_STEPS)
    return GenericRTNQuantizer(module, bits)


def naive_weight_mse(layer: nn.Module, n_bits: int) -> float:
    W2 = flatten_weight_rows(layer.weight.data, layer)
    q, s = symmetric_quantize_per_channel(W2, n_bits)
    w_hat = symmetric_dequant_per_channel(q, s).reshape(layer.weight.shape)
    return (layer.weight.float() - w_hat).pow(2).mean().item()


def measure_output_mse(layer: nn.Module, inputs: torch.Tensor | None, n_bits: int) -> float:
    if inputs is None or inputs.numel() == 0:
        return float("nan")
    qmod = build_quantized_module(layer, GenericRTNQuantizer(layer, n_bits).quantize()).to(layer.weight.device)
    if isinstance(layer, nn.Embedding):
        x = inputs[:2048].long().to(layer.weight.device)
    else:
        x = inputs[:2048].to(layer.weight.device)
    with torch.no_grad():
        y_fp = _forward_orig(layer, x)
        y_q = qmod(x)
        return (y_fp.float() - y_q.float()).pow(2).mean().item()


def run_calibration(model, processor, image, prompt, n_batches: int):
    padded, *_ = pad_info(image)
    for _ in range(n_batches):
        inputs = processor(text=prompt, images=padded, return_tensors="pt").to(DEVICE)
        inputs["pixel_values"] = inputs["pixel_values"].to(dtype=next(model.parameters()).dtype)
        with torch.no_grad():
            model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
                           max_new_tokens=64, num_beams=1, use_cache=False)
    print(f"Calibration done — {n_batches} forward passes.")


def register_calibration_hooks(model: nn.Module, quantizers: dict[str, object]):
    handles = []
    def make_hook(q):
        def hook(_module, inp, _out):
            q.add_batch(inp[0].detach())
        return hook
    module_map = dict(model.named_modules())
    for name, q in quantizers.items():
        handles.append(module_map[name].register_forward_hook(make_hook(q)))
    return handles


def collect_layer_inputs(model, layer_names, processor, image, prompt, n_batches):
    captures = {n: [] for n in layer_names}
    module_map = dict(model.named_modules())
    def make_hook(name):
        def hook(_module, inp, _out):
            x = inp[0].detach()
            if isinstance(_module, nn.Embedding):
                captures[name].append(x.cpu()[:4])
            elif isinstance(_module, (nn.Conv1d, nn.Conv2d, nn.Conv3d)):
                captures[name].append(x.cpu()[:2])
            elif x.dim() == 3:
                x = x.reshape(-1, x.shape[-1])
                captures[name].append(x.cpu()[:2048])
            else:
                captures[name].append(x.cpu()[:2048])
        return hook
    handles = [module_map[n].register_forward_hook(make_hook(n)) for n in layer_names]
    run_calibration(model, processor, image, prompt, n_batches)
    for h in handles:
        h.remove()
    for name in layer_names:
        chunks = captures[name]
        if not chunks:
            captures[name] = torch.empty(0)
        else:
            try:
                captures[name] = torch.cat(chunks, dim=0)
            except RuntimeError:
                captures[name] = chunks[0]
    return captures


def analyze_sensitivity(profiles_b, all_modules, captures) -> list[LayerProfile]:
    """Phase B LayerProfile → Phase C LayerProfile (new fields, same layer names)."""
    mod_map = dict(all_modules)
    profiles: list[LayerProfile] = []
    for inv in profiles_b:
        mod = mod_map[inv.name]
        inp = captures.get(inv.name)
        act_max = float(inp.float().abs().amax().item()) if inp is not None and inp.numel() else 0.0
        out4 = measure_output_mse(mod, inp, 4)
        out8 = measure_output_mse(mod, inp, 8)
        w4, w8 = naive_weight_mse(mod, 4), naive_weight_mse(mod, 8)
        sens = (out4 if not math.isnan(out4) else w4) * (1.0 + 0.1 * math.log1p(act_max))
        profiles.append(LayerProfile(
            name=inv.name, module_type=inv.module_type, shape=inv.shape,
            num_params=inv.num_params, protected=inv.protected,
            weight_mse_int4=w4, weight_mse_int8=w8,
            output_mse_int4=out4, output_mse_int8=out8,
            act_max=act_max, sensitivity=sens,
        ))
    profiles.sort(key=lambda p: p.sensitivity, reverse=True)
    return profiles


print("Phase C LayerProfile loaded — sensitivity fields: output_mse, act_max, sensitivity.")


In [ ]:
if "all_quantizable" not in dir() or all_quantizable is None:
    all_quantizable = iter_quantizable_modules(model, MAX_ANALYZE_LAYERS or MAX_QUANT_LAYERS)

layer_names = [n for n, _ in all_quantizable]
print(f"Capturing activations for {len(layer_names)} layers...")
captures = collect_layer_inputs(model, layer_names, processor, image, PROMPT, MAX_CALIB_BATCHES)

profiles_c: list[LayerProfile] = analyze_sensitivity(profiles_b, all_quantizable, captures)

print(f"\n{'Rank':<5} {'Sensitivity':>11} {'OutMSE i4':>11} {'OutMSE i8':>11}  Layer")
print("-" * 90)
for i, p in enumerate(profiles_c[:15], 1):
    tag = " [PROTECT]" if p.protected else ""
    print(f"{i:<5} {p.sensitivity:>11.2e} {p.output_mse_int4:>11.2e} {p.output_mse_int8:>11.2e}  {p.name.split('.')[-1]}{tag}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show = profiles_c[:min(20, len(profiles_c))]
labels = [p.name.split(".")[-1] for p in show]
axes[0].barh(labels[::-1], [p.sensitivity for p in show][::-1], color="coral")
axes[0].set_xlabel("Sensitivity score (higher = more fragile)")
axes[0].set_title("Most sensitive layers — quantize these last or use int8")

x = range(len(show))
axes[1].bar([i - 0.2 for i in x], [p.output_mse_int4 for p in show], width=0.4, label="output MSE @ int4", color="#e74c3c")
axes[1].bar([i + 0.2 for i in x], [p.output_mse_int8 for p in show], width=0.4, label="output MSE @ int8", color="#3498db")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[1].set_ylabel("Output MSE")
axes[1].set_title("Same layer — int8 usually hurts less than int4")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


---
## Phase D — Smart quantization (beat the Phase A baseline)

📎 **[Phase D — all functions map (11→14)](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_d_overview.png)**

📎 **[Phase D — mixed precision plan, apply, and OCR verify](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_d_overview.png)**

> **Question this phase answers:** *Can we get naive-level compression **without** naive-level OCR loss?*

Phase A = int4 everywhere, no plan. Phase D = **same model**, but each layer gets the right precision using Phase C rankings.

### What you will do (4 steps)

| Step | Stage | Action | Output |
|------|-------|--------|--------|
| **11** | below | Assign int4 / int8 / fp16 per layer | `profiles_d` |
| **12** | below | Load → calibrate → quantize → swap (one layer at a time) | `profiles_d_applied` |
| **13** | below | OCR detect on smart-quantized `model` | `lines` |
| **14** | below | Scorecard: fp16 vs naive (A) vs smart (D) | proof it worked |

### Phase D `LayerProfile` — plan + apply fields

📎 **[Phase D LayerProfile — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_d_layerprofile_lines.png)**

📎 **[Phase D LayerProfile — plan then apply fields visual](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_d_layerprofile_lines.png)**

| Field | Filled in | Meaning |
|-------|-----------|---------|
| `name`, `module_type`, `num_params`, `sensitivity`, `protected` | Stage 11 | From Phase C `profiles_c` |
| `bits` | Stage 11 | `int4`, `int8`, or `fp16` |
| `note` | Stage 11 | Human-readable reason |
| `weight_mse`, `output_mse`, `fp_bytes`, `q_bytes` | Stage 12 | Filled after quantize |
| `applied` | Stage 12 | `True` if layer was swapped |

**Input:** `profiles_c` (Phase C) → **`profiles_d`** (plan) → **`profiles_d_applied`** (after swap).

### Stage 11 — Mixed precision plan

Rank by `sensitivity` from Phase C. Assign bits:

| Precision | Rule (`MIXED_PRECISION = True`) |
|-----------|--------------------------------|
| **fp16** | `protected=True` **OR** top `FP16_SENSITIVE_PCT`% by sensitivity |
| **int8** | Next `INT8_MID_PCT`% by sensitivity |
| **int4** | Remaining layers (lowest sensitivity, max compression) |
| **int8** (Embedding) | Embedding layers always int8 (safer than int4 for lookup) |

Config knobs (cell 0): `FP16_SENSITIVE_PCT=15`, `INT8_MID_PCT=35`, `ALWAYS_FP16_PATTERNS`.

Set **`MIXED_PRECISION = False`** to use uniform `BITS` on all non-protected layers (ablation only).

**Method:** `QUANT_METHOD` (`gptq`, `awq`, …) is applied in Stage 12 — Stage 11 only picks **how many bits**, not the algorithm.

#### What to look for in the plan table

- **fp16 slice** should include `lm_head`, embeddings, and highest-sensitivity MLP/attention layers
- **int4 slice** should be large layers with **low** sensitivity — most of the compression lives here
- Pie chart: aim for ~60–80% params in int4, 10–20% int8, 10–20% fp16 (varies by model)


![Stage 11a — Phase D LayerProfile + build_quant_plan](attachment:nb02_stage11a_layerprofile.png)

#### Stage 11a — Phase D helpers (run this first)

Defines Phase D `LayerProfile`, `build_quant_plan()`, and `build_uniform_plan()`.

**Input:** `profiles_c` from Phase C  
**Output:** function ready to build `profiles_d` in the next cell

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Phase D — LayerProfile (mixed-precision plan + apply result per layer)
# ═══════════════════════════════════════════════════════════════
@dataclass
class LayerProfile:
    """Phase D LayerProfile — bits assignment + optional post-apply metrics."""
    name: str
    module_type: str
    num_params: int
    sensitivity: float
    protected: bool
    bits: str = "int4"
    note: str = ""
    weight_mse: float = 0.0
    output_mse: float = 0.0
    fp_bytes: int = 0
    q_bytes: int = 0
    applied: bool = False


def build_quant_plan(profiles_c: list) -> list[LayerProfile]:
    """Phase C LayerProfile → Phase D LayerProfile (assign bits)."""
    plan: list[LayerProfile] = []
    for row in profiles_c:
        plan.append(LayerProfile(
            name=row.name, module_type=row.module_type,
            num_params=row.num_params, sensitivity=row.sensitivity,
            protected=row.protected,
        ))
    for entry in plan:
        if entry.protected:
            entry.bits = "fp16"
            entry.note = "protected (head/embed/tiny layer)"
    candidates = [e for e in plan if not e.protected]
    n = len(candidates)
    if n == 0:
        return plan
    n_fp16 = max(1, round(n * FP16_SENSITIVE_PCT / 100))
    n_int8 = max(0, round(n * INT8_MID_PCT / 100))
    for i, entry in enumerate(candidates):
        if entry.module_type == "Embedding":
            entry.bits, entry.note = "int8", "Embedding — per-row int8"
            continue
        if i < n_fp16:
            entry.bits, entry.note = "fp16", f"high sensitivity (rank {i+1}/{n})"
        elif i < n_fp16 + n_int8:
            entry.bits, entry.note = "int8", "medium sensitivity — int8"
        else:
            entry.bits, entry.note = "int4", "low sensitivity — int4"
    return plan


def build_uniform_plan(profiles_c: list, bits: int) -> list[LayerProfile]:
    plan: list[LayerProfile] = []
    for row in profiles_c:
        if row.protected:
            plan.append(LayerProfile(
                name=row.name, module_type=row.module_type,
                num_params=row.num_params, sensitivity=row.sensitivity,
                protected=True, bits="fp16", note="protected",
            ))
        else:
            plan.append(LayerProfile(
                name=row.name, module_type=row.module_type,
                num_params=row.num_params, sensitivity=row.sensitivity,
                protected=False, bits=f"int{bits}", note=f"uniform int{bits}",
            ))
    return plan


print("Phase D LayerProfile loaded — plan fields: bits, note; apply fills weight_mse, applied.")


### Stage 11 — Build mixed-precision plan

📎 **[build_uniform_plan() — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_d_build_uniform_plan.png)**


📎 **[build_quant_plan() — deep line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_d_build_quant_plan_deep.png)**


📎 **[Stage 11 — why mixed precision beats naive int4](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_build_quant_plan_lines.png)**

**Run the cell below** to turn `profiles_c` rankings into `profiles_d` bit assignments.

📎 **[build_quant_plan — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_build_quant_plan_lines.png)**


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 11 — build Phase D LayerProfile list (int4 / int8 / fp16)
# ═══════════════════════════════════════════════════════════════
if MIXED_PRECISION:
    profiles_d: list[LayerProfile] = build_quant_plan(profiles_c)
else:
    profiles_d = build_uniform_plan(profiles_c, BITS)

from collections import Counter
bit_counts = Counter(p.bits for p in profiles_d)
param_by_bits = {b: sum(p.num_params for p in profiles_d if p.bits == b) for b in ["fp16", "int8", "int4"]}

print("Mixed precision plan (Phase D LayerProfile)")
print("=" * 70)
print(f"{'Bits':<8} {'Layers':>8} {'Params':>12} {'% of quant':>12}")
print("-" * 70)
for bits in ["fp16", "int8", "int4"]:
    n_layers = bit_counts.get(bits, 0)
    n_params = param_by_bits.get(bits, 0)
    pct = 100.0 * n_params / max(total_quant_params, 1)
    print(f"{bits:<8} {n_layers:>8} {n_params:>12,} {pct:>11.1f}%")

print(f"\n{'Layer':<45} {'Bits':<6} {'Sens':>9}  Reason")
print("-" * 90)
for p in profiles_d[:20]:
    short = p.name if len(p.name) <= 44 else "…" + p.name[-43:]
    print(f"{short:<45} {p.bits:<6} {p.sensitivity:>9.2e}  {p.note}")

colors = {"fp16": "#e74c3c", "int8": "#f39c12", "int4": "#27ae60"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].pie([param_by_bits[b] for b in ["int4", "int8", "fp16"]],
            labels=["int4", "int8", "fp16"],
            colors=[colors[b] for b in ["int4", "int8", "fp16"]],
            autopct="%1.0f%%", startangle=90)
axes[0].set_title("Param share by precision")
show = profiles_d[:min(18, len(profiles_d))]
axes[1].barh([p.name.split(".")[-1] for p in show][::-1],
             [p.sensitivity for p in show][::-1],
             color=[colors.get(p.bits, "gray") for p in show][::-1])
axes[1].set_xlabel("Sensitivity")
axes[1].set_title("Red=fp16  Orange=int8  Green=int4")
plt.tight_layout(); plt.show()


### Stage 12 — Apply plan: load → quantize → swap

📎 **[apply_quant_plan() — deep line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_d_apply_quant_plan_deep.png)**


📎 **[apply_quant_plan — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_apply_quant_plan_lines.png)**

![Stage 12 — one-layer-at-a-time quantize loop](attachment:nb02_stage12_apply.png)

**Run the cell below.** It walks `profiles_d` and only quantizes rows where `bits ∈ {int4, int8}`. fp16 rows are skipped.

#### One layer at a time (why?)

Calibrating all layers at once can **OOM** on Colab. We hook one layer, run `MAX_CALIB_BATCHES` OCR forwards, quantize, swap, then move on. Downstream layers see updated weights on the next pass.

#### The loop (what each print line means)

```
[i/K] language_model.layers.3.mlp.fc1
       LOAD   W (4096, 4096) fp16  |  bits=int4  method=gptq
       SWAP   → GPTQLinear  weight_mse=1.23e-04  ✓
```

| Step | Code | What happens |
|------|------|--------------|
| ① LOAD | `model.get_submodule(name)` | Fetch fp16 `nn.Linear` (or Conv / Embedding) |
| ② CALIBRATE | `register_calibration_hooks` + `run_calibration` | Collect $\mathbf{X}_\ell$ for GPTQ/AWQ/etc. |
| ③ QUANTIZE | `quantizer.quantize()` | $\mathbf{W} \to \hat{\mathbf{W}}$ using chosen `QUANT_METHOD` |
| ④ SWAP | `replace_module(model, name, qlayer)` | Graph now uses `GPTQLinear` / `AWQLinear` / … |
| ⑤ LOG | Update `LayerProfile` | `weight_mse`, `applied=True` → `profiles_d_applied` |

#### Before vs after one layer

| | Before | After swap |
|---|--------|------------|
| Module | `nn.Linear` | `GPTQLinear` (or AWQ, etc.) |
| Weights | `.weight` fp16 `[O,I]` | `.weight_q` int4 + `.weight_scales` |
| Forward | `y = xW^T + b` | `y = x · dequant(W_q)^T + b` |
| Shapes | unchanged | unchanged |

**Runtime:** Full model quant can take **10–30+ minutes** on Colab depending on layer count and `QUANT_METHOD`. GPTQ is slower than RTN; that is expected.


In [ ]:

# ═══════════════════════════════════════════════════════════════
# Stage 12 — apply_quant_plan (Phase D LayerProfile)
# ═══════════════════════════════════════════════════════════════
def apply_quant_plan(model, profiles_d: list[LayerProfile], method, captures=None) -> list[LayerProfile]:
    """Quantize int4/int8 layers; update each LayerProfile with apply metrics."""
    to_quant = [p for p in profiles_d if p.bits in ("int4", "int8")]
    if not to_quant:
        print("Nothing to quantize — all layers marked fp16.")
        return profiles_d

    name_to_mod = dict(iter_quantizable_modules(model))
    n_bits_map = {p.name: int(p.bits.replace("int", "")) for p in to_quant}
    applied: list[LayerProfile] = []

    print(f"\n── Weight update loop: {len(to_quant)} layers (one at a time) ──\n")

    for i, entry in enumerate(to_quant, 1):
        if entry.name not in name_to_mod:
            print(f"  [{i}/{len(to_quant)}] SKIP {entry.name} — not found")
            continue
        mod = name_to_mod[entry.name]
        if not isinstance(mod, QUANT_MODULE_TYPES):
            print(f"  [{i}/{len(to_quant)}] SKIP {entry.name} — already replaced")
            continue

        n_bits = n_bits_map[entry.name]
        print(f"  [{i}/{len(to_quant)}] {entry.name}")
        print(f"       LOAD   W {tuple(mod.weight.shape)} fp16  |  bits={entry.bits}  method={method}")

        quantizer = build_quantizer_for_module(method, mod, n_bits=n_bits)
        handles = register_calibration_hooks(model, {entry.name: quantizer})
        run_calibration(model, processor, image, PROMPT, MAX_CALIB_BATCHES)
        for h in handles:
            h.remove()

        state = quantizer.quantize()
        qlayer = build_quantized_module(mod, state).to(DEVICE)
        w_mse = layer_weight_mse(mod, qlayer)
        o_mse = layer_output_mse(mod, qlayer, captures.get(entry.name) if captures else None)
        replace_module(model, entry.name, qlayer)

        entry.weight_mse = w_mse
        entry.output_mse = o_mse
        entry.fp_bytes = layer_num_params(mod) * 2
        entry.q_bytes = qlayer.storage_bytes()
        entry.applied = True
        applied.append(entry)
        print(f"       SWAP   → {type(qlayer).__name__}  weight_mse={w_mse:.2e}  ✓\n")
        if i % 10 == 0 or i == len(to_quant):
            print(f"  progress: {i}/{len(to_quant)} layers done")

    return applied


to_quant = [p for p in profiles_d if p.bits in ("int4", "int8")]
to_keep = [p for p in profiles_d if p.bits == "fp16"]
print(f"Plan: quantize {len(to_quant)} layers  ·  keep {len(to_keep)} at fp16  ·  method={QUANT_METHOD.upper()}")
for p in to_quant[:8]:
    print(f"  {p.bits:>4}  {p.name}")
if len(to_quant) > 8:
    print(f"  ... +{len(to_quant)-8} more")

t0 = time.time()
profiles_d_applied: list[LayerProfile] = apply_quant_plan(model, profiles_d, QUANT_METHOD, captures)
elapsed = time.time() - t0

if profiles_d_applied:
    avg_mse = sum(p.weight_mse for p in profiles_d_applied) / len(profiles_d_applied)
    total_fp = sum(p.fp_bytes for p in profiles_d_applied)
    total_q = sum(p.q_bytes for p in profiles_d_applied)
    print(f"\nFinished in {elapsed:.1f}s")
    print(f"Average weight MSE on quant layers: {avg_mse:.2e}")
    print(f"Quantized storage: {total_q/1024:.1f} KB  vs  {total_fp/1024:.1f} KB fp16 before")
    print(f"Compression on quant layers: {total_fp/max(total_q,1):.2f}×")
else:
    print("No layers were quantized.")


### Stage 12b — Weight error chart

![Stage 12b — per-layer weight MSE after apply](attachment:nb02_stage12b_weight_chart.png)

Green = int4, orange = int8. High bars = layers where quantization hurt weights most — cross-check with Phase C sensitivity.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 12b — weight error chart (Phase D LayerProfile, applied layers)
# ═══════════════════════════════════════════════════════════════
if profiles_d_applied:
    colors = {"int4": "#27ae60", "int8": "#f39c12"}
    names = [p.name.split(".")[-1] for p in profiles_d_applied]
    mses = [p.weight_mse for p in profiles_d_applied]
    bar_c = [colors.get(p.bits, "steelblue") for p in profiles_d_applied]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(len(mses)), mses, color=bar_c)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Weight MSE")
    ax.set_title(f"{QUANT_METHOD.upper()} — Phase D LayerProfile (green=int4, orange=int8)")
    plt.tight_layout(); plt.show()


---
### Stage 13 — Does OCR still work?

📎 **[Stage 13 — OCR detect verification with bounding boxes](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_run_florence_detect_lines.png)**

**Run the cell below** on the **smart-quantized** `model` from Stage 12 (not `model_naive`).

We care about **task quality**, not average weight MSE. A model can have low weight error everywhere and still miss text lines.

#### What the cell does

1. Run `<OCR_WITH_REGION>` detect on the calibration page
2. Print detected lines (first 8) + total count → saved as `lines`
3. Draw green boxes on the image — visual sanity check

#### How to judge success (practical checklist)

| Check | Pass | Fail |
|-------|------|------|
| Line count vs fp16 (`baseline_ocr`) | within ±2 lines | large drop |
| Overlay boxes | align with text on page | missing regions or garbage |
| Readable text in printout | words match the document | repeated tokens / empty |

#### IoU (optional formal metric)

For boxes $B$ and $\hat{B}$ from fp16 vs quant model:

$$\text{IoU}(B, \hat{B}) = \frac{|B \cap \hat{B}|}{|B \cup \hat{B}|}$$

**Target:** median IoU $\ge 0.85$ and $|L_{\text{quant}} - L_{\text{fp}}| \le 2$ on the calibration page.

OCR is a **set of rectangles + strings** — pixel MSE on weights does not predict box overlap. That is why Phase C measured **output MSE**, and Phase D protects high-sensitivity layers.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 13 — run OCR detect on smart-quantized model
# ═══════════════════════════════════════════════════════════════
lines, infer_s = run_florence_detect(image, processor, model, DEVICE)
print(f"Took {infer_s:.2f}s — found {len(lines)} text lines")
for line in lines[:8]:
    print(f"  • {line['text'][:70]}")
if len(lines) > 8:
    print(f"  ... +{len(lines)-8} more lines")

vis = image.copy(); draw = ImageDraw.Draw(vis)
for line in lines:
    b = line["bbox"]
    draw.rectangle(b, outline="lime", width=2)
plt.figure(figsize=(8, 6)); plt.imshow(vis)
plt.title(f"Smart {QUANT_METHOD.upper()} — detect overlay"); plt.axis("off"); plt.show()


### Stage 14 — Compare: naive vs smart vs fp16

📎 **[Stage 14 — scorecard gap visual](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_stage14_scorecard_gap.png)**

📎 **[Stage 14 — scorecard: fp16 vs naive int4 vs smart quant](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_stage14_scorecard_gap.png)**

**Run the cell below.** This is the **payoff** of the whole A→D pipeline.

#### Scorecard

| Approach | Model | Plan | What we measure |
|----------|-------|------|-----------------|
| **fp16** | fresh load | none | Reference line count |
| **naive int4** | `model_naive` (Phase A) | int4 everywhere | `baseline_ocr` from Stage 8c |
| **smart quant** | `model` (Phase D) | mixed int4/int8/fp16 | `len(lines)` from Stage 13 |

#### How to read the result

| Outcome | Meaning | Next step |
|---------|---------|-----------|
| smart ≥ fp16 (±2 lines) | Quantization succeeded | Export in notebook 03 |
| smart > naive lines | Mixed plan **helped** vs Phase A | Good — sensitivity routing works |
| smart ≈ naive | Plan did not recover quality | Raise `FP16_SENSITIVE_PCT` or try `awq` / `smoothquant` |
| smart < naive | Something wrong | Re-run Phase C captures; check `QUANT_METHOD` |

#### Full pipeline recap (A → D)

```
Phase A  profiles_a     naive int4 all layers     → baseline_ocr
Phase B  profiles_b     who is big / protected?
Phase C  profiles_c     who is sensitive?
Phase D  profiles_d     assign bits → apply → OCR → compare
```

Each phase uses its **own `LayerProfile`** definition. Data flows through `profiles_a` → `profiles_b` → `profiles_c` → `profiles_d` — never one shared struct.

**Optional next:** Phase E runs per-method demos and a 5-way OCR compare if you want to pick the best `QUANT_METHOD`.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 14 — naive vs smart vs fp16 scorecard
# ═══════════════════════════════════════════════════════════════
smart_lines = len(lines)
ref = baseline_ocr

print(f"{'Approach':<18} {'Lines':>6} {'Δ vs fp16':>10} {'Infer(s)':>10}")
print("-" * 48)
print(f"{'fp16':<18} {ref['fp16_lines']:>6} {'—':>10} {ref['fp16_infer_s']:>10.2f}")
print(f"{'naive int4':<18} {ref['naive_lines']:>6} {ref['naive_lines']-ref['fp16_lines']:>+10d} {ref['naive_infer_s']:>10.2f}")
print(f"{'smart quant':<18} {smart_lines:>6} {smart_lines-ref['fp16_lines']:>+10d} {infer_s:>10.2f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
labels = ["fp16", "naive int4", f"smart {QUANT_METHOD}"]
counts = [ref["fp16_lines"], ref["naive_lines"], smart_lines]
colors = ["#3498db", "#e74c3c", "#27ae60"]
ax.bar(labels, counts, color=colors)
ax.set_ylabel("Detected lines")
ax.set_title("Phase A naive vs Phase D smart — OCR line count")
plt.tight_layout(); plt.show()

if smart_lines >= ref["naive_lines"]:
    print("\n✓ Smart quant recovered or improved line count vs naive int4.")
else:
    print("\n⚠ Smart quant still below naive — try a different QUANT_METHOD or raise FP16_SENSITIVE_PCT.")


---
## Phase E — Method deep dives (optional)

![Stage 16 — compare all five quant methods on OCR](attachment:nb02_phase_e_layerprofile.png)

Stages 2–6 already explained the math. These demos run **one layer each** so you can see GPTQ vs AWQ vs RTN side by side.

Phase E defines its own **`LayerProfile`** (method, lines, quant_s). Skip to **Stage 16** if you only care about the full OCR comparison.

### Stage 15 — Per-method demos (GPTQ → AWQ → SmoothQuant → SpinQuant → ConvRot)

Each demo quantizes the same layer and reports **output MSE vs round-to-nearest (RTN)** on real OCR calibration data:

| Sub-stage | Method | What the demo shows |
|-----------|--------|---------------------|
| **15a** | GPTQ | Hessian-aware column quant beats naive RTN |
| **15b** | AWQ | Activation-aware scales beat naive RTN |
| **15c** | SmoothQuant | Outlier migration + alpha sweep |
| **15d** | SpinQuant | Learned Givens rotation beats RTN |
| **15e** | ConvRot | Group-wise RHT beats RTN |


### Stage 15a — GPTQ demo (Hessian-aware vs RTN)

![GPTQ demo — Hessian-aware vs RTN](attachment:nb02_stage02_gptq.png)

GPTQ uses the Hessian $\mathbf{H} = \mathbf{X}^\top\mathbf{X}$ to quantize columns in order of importance:

$$
\text{MSE}_{\text{GPTQ}} < \text{MSE}_{\text{RTN}} \quad \text{(Hessian weights high-impact directions)}
$$


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 15a — GPTQ demo (Hessian-aware vs RTN)
# ═══════════════════════════════════════════════════════════════
_demo_cands = [(n, m) for n, m in iter_linear_modules(model, 8) if m.weight.numel() >= MIN_PARAMS_TO_QUANT]

def _rtn_output_mse(layer, inputs, n_bits=4):
    """RTN baseline output MSE for demo comparisons."""
    if inputs is None or inputs.numel() == 0:
        return float("nan")
    W2 = flatten_weight_rows(layer.weight.data, layer)
    q, s = symmetric_quantize_per_channel(W2, n_bits)
    w_hat = symmetric_dequant_per_channel(q, s).reshape(layer.weight.shape)
    x = inputs[:512].to(layer.weight.device)
    with torch.no_grad():
        y_fp = _forward_orig(layer, x)
        if isinstance(layer, nn.Linear):
            y_rtn = F.linear(x, w_hat.to(x.dtype), layer.bias)
        else:
            y_rtn = y_fp  # fallback
        return (y_fp.float() - y_rtn.float()).pow(2).mean().item()

if _demo_cands and captures:
    _dn, _dl = _demo_cands[0]
    _xin = captures.get(_dn, torch.empty(0))
    if _xin.numel() > 0:
        mse_rtn = _rtn_output_mse(_dl, _xin, 4)
        _gq = GPTQQuantizer(_dl, n_bits=4)
        _h = _dl.register_forward_hook(lambda m, inp, out, q=_gq: q.add_batch(inp[0].detach()))
        run_calibration(model, processor, image, PROMPT, 2)
        _h.remove()
        _ql = QuantizedLinear(_dl.in_features, _dl.out_features, _gq.quantize()).to(DEVICE)
        mse_gptq = layer_output_mse(_dl, _ql, _xin)
        print(f"GPTQ demo — {_dn}  shape={tuple(_dl.weight.shape)}")
        print(f"  RTN output MSE : {mse_rtn:.4e}")
        print(f"  GPTQ output MSE: {mse_gptq:.4e}")
        print(f"  GPTQ improvement: {(1 - mse_gptq/max(mse_rtn,1e-12))*100:.1f}% vs RTN")
    else:
        print("GPTQ demo skipped — no calibration inputs.")
else:
    print("GPTQ demo skipped — run sensitivity capture first.")


### Stage 15b — AWQ demo (activation-aware scales vs RTN)

![AWQ demo — activation-aware scales vs RTN](attachment:nb02_stage03_awq.png)

AWQ searches per-channel scales $s_j$ that minimize activation-weighted error:

$$
\min_{s} \|\mathbf{X}\mathbf{W}^\top - \mathbf{X} \cdot Q(\mathbf{W} \odot s)\|^2
$$


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 15b — AWQ demo (activation-aware scales vs RTN)
# ═══════════════════════════════════════════════════════════════
_demo_cands = [(n, m) for n, m in iter_linear_modules(model, 8) if m.weight.numel() >= MIN_PARAMS_TO_QUANT]

def _rtn_output_mse(layer, inputs, n_bits=4):
    """RTN baseline output MSE for demo comparisons."""
    if inputs is None or inputs.numel() == 0:
        return float("nan")
    W2 = flatten_weight_rows(layer.weight.data, layer)
    q, s = symmetric_quantize_per_channel(W2, n_bits)
    w_hat = symmetric_dequant_per_channel(q, s).reshape(layer.weight.shape)
    x = inputs[:512].to(layer.weight.device)
    with torch.no_grad():
        y_fp = _forward_orig(layer, x)
        if isinstance(layer, nn.Linear):
            y_rtn = F.linear(x, w_hat.to(x.dtype), layer.bias)
        else:
            y_rtn = y_fp  # fallback
        return (y_fp.float() - y_rtn.float()).pow(2).mean().item()

if _demo_cands and captures:
    _dn, _dl = _demo_cands[0]
    _xin = captures.get(_dn, torch.empty(0))
    if _xin.numel() > 0:
        mse_rtn = _rtn_output_mse(_dl, _xin, 4)
        _aq = AWQQuantizer(_dl, n_bits=4)
        _h = _dl.register_forward_hook(lambda m, inp, out, q=_aq: q.add_batch(inp[0].detach()))
        run_calibration(model, processor, image, PROMPT, 2)
        _h.remove()
        _ql = QuantizedLinear(_dl.in_features, _dl.out_features, _aq.quantize()).to(DEVICE)
        mse_awq = layer_output_mse(_dl, _ql, _xin)
        print(f"AWQ demo — {_dn}  shape={tuple(_dl.weight.shape)}")
        print(f"  RTN output MSE : {mse_rtn:.4e}")
        print(f"  AWQ output MSE : {mse_awq:.4e}")
        print(f"  AWQ improvement: {(1 - mse_awq/max(mse_rtn,1e-12))*100:.1f}% vs RTN")
    else:
        print("AWQ demo skipped — no calibration inputs.")
else:
    print("AWQ demo skipped — run Stage 15a first.")


### Stage 15c — SmoothQuant demo (alpha sweep + act ranges)

![SmoothQuant demo — alpha sweep and activation ranges](attachment:nb02_stage04_smoothquant.png)

This demo sweeps $\alpha \in \{0.3, 0.5, 0.7\}$ and measures weight/output MSE on real calibration data.

### Smoothing (matches Stage 4 + `SmoothQuantQuantizer`)

Per input channel $j$, calibration gives activation max $A_j = \max_m |X_j^{(m)}|$ and weight max $W_{\max,j} = \max_i |W_{ij}|$. The migration scale is:

$$
s_j(\alpha) = \frac{A_j^{\alpha}}{W_{\max,j}^{\,1-\alpha}}, \qquad
W'_{ij} = \frac{W_{ij}}{s_j}, \qquad X'_{ij} = X_{ij} \cdot s_j
$$

**Theorem (exact equivalence):** $\mathbf{X}\mathbf{W}^\top = \mathbf{X}'{\mathbf{W}'}^\top$ because $(X_{ij} s_j)(W_{ij}/s_j) = X_{ij} W_{ij}$.

### Effect on dynamic ranges

Post-smooth per-channel activation and weight magnitudes:

$$
|X'_j|_{\max} = A_j \cdot s_j(\alpha) = \frac{A_j^{\,1+\alpha}}{W_{\max,j}^{\,1-\alpha}}, \qquad
|W'_{j}|_{\max} = \frac{W_{\max,j}}{s_j(\alpha)} = \frac{W_{\max,j}^{\,2-\alpha}}{A_j^{\alpha}}
$$

**Limits:**
- $\alpha = 0$: $s_j = 1$ $\Rightarrow$ $|X'_j|_{\max} = A_j$, $|W'_{j}|_{\max} = W_{\max,j}$ (no migration)
- $\alpha \to 1$: outliers move from activations into weights; $|X'_j|_{\max}$ shrinks relative to $A_j^2$, $|W'_{j}|_{\max}$ grows relative to $W_{\max,j}/A_j$

### Inference (what `SmoothQuantLinear.forward` does)

$$
\mathbf{y} = (\mathbf{x} \odot \mathbf{s}) \cdot \text{dequant}(Q(\mathbf{W}'))^\top + \mathbf{b}
$$

The demo plots $\text{MSE}_{\text{out}}$ vs $\alpha$ and compares global activation range before/after scaling: $\max_{j,m}|X_j^{(m)}|$ vs $\max_{j,m}|X_j^{(m)} s_j|$.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 15c — SmoothQuant demo (alpha sweep + act ranges)
# ═══════════════════════════════════════════════════════════════
_demo_cands = [(n, m) for n, m in iter_linear_modules(model, 8) if m.weight.numel() >= MIN_PARAMS_TO_QUANT]

if _demo_cands and captures:
    _dn, demo_layer = _demo_cands[0]
    demo_inputs = captures.get(_dn, torch.empty(0))
    print(f"SmoothQuant demo layer: {_dn}  shape={tuple(demo_layer.weight.shape)}")

    alpha_results = []
    for alpha in [0.3, 0.5, 0.7]:
        sq = SmoothQuantQuantizer(demo_layer, n_bits=4, alpha=alpha)
        h2 = demo_layer.register_forward_hook(lambda m, inp, out, q=sq: q.add_batch(inp[0].detach()))
        run_calibration(model, processor, image, PROMPT, 2)
        h2.remove()
        state = sq.quantize()
        ql = QuantizedLinear(demo_layer.in_features, demo_layer.out_features, state).to(DEVICE)
        act_before = float(demo_inputs.abs().amax()) if demo_inputs.numel() else 0.0
        act_after = float((demo_inputs * state.act_scales).abs().amax()) if demo_inputs.numel() else 0.0
        alpha_results.append({
            "alpha": alpha,
            "weight_mse": layer_weight_mse(demo_layer, ql),
            "output_mse": layer_output_mse(demo_layer, ql, demo_inputs),
            "act_max_before": act_before,
            "act_max_after": act_after,
        })

    print(f"\n{'Alpha':<8} {'Weight MSE':>12} {'Output MSE':>12} {'Act max before':>14} {'Act max after':>14}")
    print("-" * 65)
    for r in alpha_results:
        print(f"{r['alpha']:<8.1f} {r['weight_mse']:>12.2e} {r['output_mse']:>12.2e} {r['act_max_before']:>14.2f} {r['act_max_after']:>14.2f}")

    best = min(alpha_results, key=lambda x: x["output_mse"])
    print(f"\nBest alpha for this layer (lowest output MSE): {best['alpha']}  — set SMOOTHQUANT_ALPHA in config")

    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].bar([str(r["alpha"]) for r in alpha_results], [r["output_mse"] for r in alpha_results], color="#C44E52")
    ax[0].set_title("SmoothQuant output MSE by alpha"); ax[0].set_xlabel("alpha")
    ax[1].bar(["before", "after"], [alpha_results[1]["act_max_before"], alpha_results[1]["act_max_after"]], color=["#e74c3c", "#27ae60"])
    ax[1].set_title("Activation range @ alpha=0.5"); plt.tight_layout(); plt.show()
else:
    print("SmoothQuant demo skipped — run Stage 9 capture first.")


### Stage 15d — SpinQuant demo (learned Givens rotation vs RTN)

![SpinQuant demo — learned Givens rotation vs RTN](attachment:nb02_stage05_spinquant.png)

$$
\text{MSE}_{\text{Spin}} = \|\mathbf{X}\mathbf{W}^\top - Q(\mathbf{X}\mathbf{R})\, Q(\mathbf{W}\mathbf{R})^\top\|^2 < \text{MSE}_{\text{RTN}}
$$


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 15d — SpinQuant demo (learned Givens rotation vs RTN)
# ═══════════════════════════════════════════════════════════════
_demo_cands = [(n, m) for n, m in iter_linear_modules(model, 8) if m.weight.numel() >= MIN_PARAMS_TO_QUANT]

def _rtn_output_mse(layer, inputs, n_bits=4):
    """RTN baseline output MSE for demo comparisons."""
    if inputs is None or inputs.numel() == 0:
        return float("nan")
    W2 = flatten_weight_rows(layer.weight.data, layer)
    q, s = symmetric_quantize_per_channel(W2, n_bits)
    w_hat = symmetric_dequant_per_channel(q, s).reshape(layer.weight.shape)
    x = inputs[:512].to(layer.weight.device)
    with torch.no_grad():
        y_fp = _forward_orig(layer, x)
        if isinstance(layer, nn.Linear):
            y_rtn = F.linear(x, w_hat.to(x.dtype), layer.bias)
        else:
            y_rtn = y_fp  # fallback
        return (y_fp.float() - y_rtn.float()).pow(2).mean().item()

if _demo_cands and captures:
    _dn, _dl = _demo_cands[0]
    _xin = captures.get(_dn, torch.empty(0))
    if _xin.numel() > 0:
        _sq = SpinQuantQuantizer(_dl, n_bits=4)
        _h = _dl.register_forward_hook(lambda m, inp, out, q=_sq: q.add_batch(inp[0].detach()))
        run_calibration(model, processor, image, PROMPT, 2)
        _h.remove()
        _ql = QuantizedLinear(_dl.in_features, _dl.out_features, _sq.quantize()).to(DEVICE)
        mse_spin = layer_output_mse(_dl, _ql, _xin)
        print(f"SpinQuant demo — {_dn}  shape={tuple(_dl.weight.shape)}")
        print(f"  RTN output MSE      : {mse_rtn:.4e}")
        print(f"  SpinQuant output MSE: {mse_spin:.4e}")
        print(f"  SpinQuant improvement: {(1 - mse_spin/max(mse_rtn,1e-12))*100:.1f}% vs RTN")
    else:
        print("SpinQuant demo skipped — no calibration inputs.")
else:
    print("SpinQuant demo skipped — run Stage 15a first.")


### Stage 15e — ConvRot demo (group-wise RHT vs RTN)

![ConvRot demo — group-wise RHT vs RTN](attachment:nb02_stage06_convrot.png)

ConvRot applies a fixed Regular Hadamard Transform per group — no training needed:

$$
\mathbf{Y} = \sum_i \text{RHT}(\mathbf{X}_i)\,\text{RHT}(\mathbf{W}_i)^\top, \quad O(K) \text{ complexity}
$$


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stage 15e — ConvRot demo (group-wise RHT vs RTN)
# ═══════════════════════════════════════════════════════════════
_demo_cands = [(n, m) for n, m in iter_linear_modules(model, 8) if m.weight.numel() >= MIN_PARAMS_TO_QUANT]

def _rtn_output_mse(layer, inputs, n_bits=4):
    """RTN baseline output MSE for demo comparisons."""
    if inputs is None or inputs.numel() == 0:
        return float("nan")
    W2 = flatten_weight_rows(layer.weight.data, layer)
    q, s = symmetric_quantize_per_channel(W2, n_bits)
    w_hat = symmetric_dequant_per_channel(q, s).reshape(layer.weight.shape)
    x = inputs[:512].to(layer.weight.device)
    with torch.no_grad():
        y_fp = _forward_orig(layer, x)
        if isinstance(layer, nn.Linear):
            y_rtn = F.linear(x, w_hat.to(x.dtype), layer.bias)
        else:
            y_rtn = y_fp  # fallback
        return (y_fp.float() - y_rtn.float()).pow(2).mean().item()

if _demo_cands and captures:
    _dn, _dl = _demo_cands[0]
    _xin = captures.get(_dn, torch.empty(0))
    if _xin.numel() > 0:
        _cr = ConvRotQuantizer(_dl, n_bits=4)
        _st_cr = _cr.quantize()
        _ql_cr = QuantizedLinear(_dl.in_features, _dl.out_features, _st_cr).to(DEVICE)
        mse_convrot = layer_output_mse(_dl, _ql_cr, _xin)
        print(f"ConvRot demo — {_dn}  shape={tuple(_dl.weight.shape)}  group={CONVROT_GROUP_SIZE}")
        print(f"  RTN output MSE    : {mse_rtn:.4e}")
        print(f"  ConvRot output MSE: {mse_convrot:.4e}")
        print(f"  ConvRot improvement: {(1 - mse_convrot/max(mse_rtn,1e-12))*100:.1f}% vs RTN")
    else:
        print("ConvRot demo skipped — no calibration inputs.")
else:
    print("ConvRot demo skipped — run Stage 15a first.")


![Phase E LayerProfile — fair 5-method OCR compare](attachment:nb02_phase_e_layerprofile.png)

---
### Stage 16 — Compare all five quant methods on full OCR

![Stage 16 — five-method OCR comparison](attachment:nb02_phase_e_layerprofile.png)

Controlled experiment with fresh θ per method.

📎 **[Phase E LayerProfile — line by line](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phase_e_layerprofile_lines.png)**

Fresh $\theta^{(m)}$ per method:

$$
m \in \{\text{gptq, awq, smoothquant, spinquant, convrot}\}
$$

| Method | Core idea |
|--------|-----------|
| **GPTQ** | Hessian-aware column quant |
| **AWQ** | Activation-aware scale search |
| **SmoothQuant** | Outlier migration $s_j$ |
| **SpinQuant** | Learned Givens $\mathbf{R}$ |
| **ConvRot** | Group-wise RHT, $O(K)$ |

Controlled by **`RUN_ALL_METHODS_OCR`** (default `True`).


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Phase E — LayerProfile (one row per quant method in OCR compare)
# ═══════════════════════════════════════════════════════════════
@dataclass
class LayerProfile:
    """Phase E LayerProfile — OCR outcome after smart quant with one method."""
    method: str
    lines: int
    quant_s: float
    infer_s: float
    sample: list[str]


def load_fresh_model():
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, trust_remote_code=True, torch_dtype=dtype, attn_implementation="eager",
    ).to(DEVICE)
    m.eval()
    return m


def run_quant_ocr(method: str) -> LayerProfile:
    m = load_fresh_model()
    t0 = time.time()
    apply_quant_plan(m, profiles_d, method, captures)
    quant_s = time.time() - t0
    lines, infer_s = run_florence_detect(image, processor, m, DEVICE)
    del m
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return LayerProfile(
        method=method, lines=len(lines), quant_s=quant_s, infer_s=infer_s,
        sample=[l["text"][:40] for l in lines[:3]],
    )


METHODS = ["gptq", "awq", "smoothquant", "spinquant", "convrot"]
METHOD_COLORS = {
    "gptq": "#4C72B0", "awq": "#55A868", "smoothquant": "#C44E52",
    "spinquant": "#8172B3", "convrot": "#CCB974",
}

if RUN_ALL_METHODS_OCR:
    print("Running full OCR pipeline for GPTQ, AWQ, SmoothQuant, SpinQuant, ConvRot...\n")
    profiles_e: list[LayerProfile] = []
    for method in METHODS:
        print(f"--- {method.upper()} ---")
        r = run_quant_ocr(method)
        profiles_e.append(r)
        print(f"  quant {r.quant_s:.1f}s  infer {r.infer_s:.1f}s  lines={r.lines}")
        for t in r.sample:
            print(f"    • {t}")
        print()

    print(f"{'Method':<14} {'Lines':>6} {'Quant(s)':>10} {'Infer(s)':>10}")
    print("-" * 44)
    for r in profiles_e:
        print(f"{r.method:<14} {r.lines:>6} {r.quant_s:>10.1f} {r.infer_s:>10.1f}")

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar([r.method for r in profiles_e], [r.lines for r in profiles_e],
           color=[METHOD_COLORS[r.method] for r in profiles_e])
    ax.set_ylabel("Detected lines")
    ax.set_title("Phase E LayerProfile — five quant methods")
    plt.tight_layout(); plt.show()
else:
    print("Set RUN_ALL_METHODS_OCR = True to compare all five quant methods on full OCR.")


# Bonus: single-layer MSE compare (all 5 methods)
if COMPARE_LAYER_MSE:
    print("\n" + "=" * 60)
    print("Single-layer MSE compare (weight + output, all 5 methods)")
    model_cmp = load_fresh_model()
    name0, layer0 = iter_linear_modules(model_cmp, 1)[0]
    cap0 = {}
    h = layer0.register_forward_hook(
        lambda m, inp, out: cap0.setdefault("x", []).append(inp[0].detach().reshape(-1, inp[0].shape[-1]).cpu())
    )
    run_calibration(model_cmp, processor, image, PROMPT, MAX_CALIB_BATCHES)
    h.remove()
    inp0 = torch.cat(cap0["x"], dim=0) if cap0.get("x") else torch.empty(0)
    print(f"Layer: {name0}\n")
    print(f"{'Method':<14} {'Weight MSE':>12} {'Output MSE':>12}")
    print("-" * 42)
    for method in METHODS:
        q = build_quantizer(method, layer0)
        q.quantize()
        ql = build_quantized_module(layer0, q.quantize()).to(DEVICE)
        wm = layer_weight_mse(layer0, ql)
        om = layer_output_mse(layer0, ql, inp0 if inp0.numel() else None)
        print(f"{method:<14} {wm:>12.2e} {om:>12.2e}")
    del model_cmp


📎 **[Phases A→D journey recap](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_phases_abcd_journey.png)**

📎 **[Five LayerProfile classes — one per phase](https://github.com/Gaurav14cs17/VLM_Quant/raw/main/assets/nb02/wb_why_five_layerprofiles.png)**


---
## What we did — Phase A → D recap

Each phase defines its **own `LayerProfile`** (same class name, different fields). Data flows through named lists — never one shared struct.

| Phase | Question answered | `LayerProfile` fields | Variable |
|-------|-------------------|----------------------|----------|
| **A** | What if int4 everywhere? | `name, bits, weight_mse, fp_bytes, q_bytes` | `profiles_a` |
| **B** | Who is big / protected? | `name, shape, num_params, protected, tiny` | `profiles_b` |
| **C** | Who breaks under int4? | `output_mse, sensitivity, act_max` | `profiles_c` |
| **D** | Smart plan + apply | `bits, note, weight_mse, applied` | `profiles_d` |
| **E** | Which method wins? | `method, lines, quant_s, infer_s` | `profiles_e` |

```
A: naive int4  →  baseline_ocr
B: inventory   →  profiles_b
C: sensitivity →  profiles_c
D: plan+apply  →  beat baseline_ocr  ✓
```

**Next:** [03 — Mobile Deployment](03_ocr_pipeline_mobile.ipynb) — pack `profiles_d_applied` weights for on-device OCR.
